# CSA17 Artificial Intelligence - Assignment 5 (CO5)
Athisaya U - 192571001

Each module is written with `%%writefile` and imported. Fixed seeds throughout. Raw data goes to `data/`, figures to `figures/`.

In [1]:
import csv, os, json, math, time, random, platform
import numpy as np
os.makedirs('data', exist_ok=True); os.makedirs('figures', exist_ok=True)
def save_csv(name, rows):
    with open(os.path.join('data', name), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
import sklearn, pandas
print('Python', platform.python_version(), '| NumPy', np.__version__, '| pandas', pandas.__version__, '| scikit-learn', sklearn.__version__,
      '| CPU cores', os.cpu_count())

Python 3.13.18 | NumPy 2.5.3 | pandas 3.0.5 | scikit-learn 1.9.1 | CPU cores 2


## Task B1 - drone agent (Task A1 design)

In [2]:
%%writefile u5_drone.py
# Task A1 / B1 - layered (hybrid) parcel-delivery drone agent and its grid simulator.
# The agent (LayeredDroneAgent) only sees percepts; the world (DeliveryWorld) owns the truth.
import heapq

MOVES = {'N': (0, 1), 'S': (0, -1), 'E': (1, 0), 'W': (-1, 0)}
MOVE_COST, HOVER_COST, HEADWIND_EXTRA = 1.0, 0.5, 0.5


# ------------------------------------------------------------------ simulator (the truth)
class DeliveryWorld:
    """W x H grid. no_fly is published (known to the agent); obstacles appear at given times and are
    only detectable by the drone's proximity sensor (range 1). Wind drifts the drone by one cell
    on scripted steps or with probability p_drift."""

    def __init__(self, W, H, base, goal, pads, no_fly=(), obstacles=None, wind=(0, 0),
                 drift_steps=(), p_drift=0.0, battery=30.0, seed=0, max_steps=200):
        import random
        self.W, self.H, self.base, self.goal = W, H, base, goal
        self.pads = [base] + [p for p in pads if p != base]
        self.no_fly = set(no_fly)
        self.obstacles = dict(obstacles or {})         # cell -> time it appears
        self.wind, self.drift_steps, self.p_drift = wind, set(drift_steps), p_drift
        self.rng = random.Random(seed)
        self.pos, self.battery, self.t = base, battery, 0
        self.delivered = self.crashed = self.landed = False
        self.max_steps, self.log = max_steps, []

    def blocked(self, c):
        x, y = c
        if not (0 <= x < self.W and 0 <= y < self.H): return True
        if c in self.no_fly: return True
        return c in self.obstacles and self.obstacles[c] <= self.t

    def percept(self):
        x, y = self.pos
        near = {d: self.blocked((x + dx, y + dy)) for d, (dx, dy) in MOVES.items()}
        return dict(t=self.t, pos=self.pos, battery=round(self.battery, 2), blocked=near,
                    wind=self.wind, at_goal=self.pos == self.goal, at_pad=self.pos in self.pads)

    def execute(self, action):
        """Apply one action; returns a short event string. Physics, not decisions."""
        event = ''
        if action in MOVES:
            dx, dy = MOVES[action]
            cost = MOVE_COST + (HEADWIND_EXTRA if (dx, dy) == (-self.wind[0], -self.wind[1]) and self.wind != (0, 0) else 0)
            self.battery -= cost
            nxt = (self.pos[0] + dx, self.pos[1] + dy)
            if self.blocked(nxt):
                self.crashed, event = True, 'COLLISION'
            else:
                self.pos = nxt
                drift = self.t in self.drift_steps or (self.p_drift > 0 and self.rng.random() < self.p_drift)
                if drift and self.wind != (0, 0):
                    d = (self.pos[0] + self.wind[0], self.pos[1] + self.wind[1])
                    if not self.blocked(d):
                        self.pos, event = d, 'drifted'
        elif action == 'HOVER':
            self.battery -= HOVER_COST
        elif action == 'DROP':
            if self.pos == self.goal: self.delivered, event = True, 'parcel delivered'
        elif action == 'LAND':
            self.landed = True
            event = 'landed at pad' if self.pos in self.pads else 'EMERGENCY landing'
        if self.battery <= 0 and not self.landed:
            self.crashed, event = True, 'BATTERY EXHAUSTED'
        self.t += 1
        return event

    def done(self):
        return self.crashed or self.landed or self.t >= self.max_steps


# ------------------------------------------------------------------ agent (decisions only)
def astar(start, target, W, H, blocked):
    """A* on the 4-connected grid with Manhattan heuristic; returns list of moves or None."""
    if start == target: return []
    h = lambda c: abs(c[0] - target[0]) + abs(c[1] - target[1])
    frontier, best, parent = [(h(start), 0, start)], {start: 0}, {start: None}
    order = ['N', 'E', 'S', 'W']                       # fixed expansion order = deterministic ties
    while frontier:
        f, g, c = heapq.heappop(frontier)
        if c == target:
            path = []
            while parent[c]: c, m = parent[c]; path.append(m)
            return path[::-1]
        if g > best[c]: continue
        for m in order:
            dx, dy = MOVES[m]; n = (c[0] + dx, c[1] + dy)
            if not (0 <= n[0] < W and 0 <= n[1] < H) or n in blocked: continue
            if g + 1 < best.get(n, 1e9):
                best[n], parent[n] = g + 1, (c, m)
                heapq.heappush(frontier, (g + 1 + h(n), g + 1, n))
    return None


class LayeredDroneAgent:
    """Two layers + arbiter. Safety layer (reactive) proposes an action only when a hazard exists;
    otherwise the deliberative layer (mission phase + A* plan) acts. Safety > mission > economy."""

    def __init__(self, W, H, goal, pads, no_fly, battery, reserve=3.0, critical_margin=1.0, use_safety=True, wind_aware=False):
        self.W, self.H, self.goal, self.pads = W, H, goal, list(pads)
        self.known_blocked = set(no_fly)                 # internal map: no-fly + discovered obstacles
        self.battery, self.reserve, self.critical = battery, reserve, critical_margin
        self.phase, self.plan, self.target = 'TO_GOAL', [], goal
        self.expected, self.pos, self.delivered = None, None, False
        self.use_safety = use_safety                    # False = ablation: deliberative layer only
        self.wind_aware, self.wind = wind_aware, (0, 0)  # True = price headwind moves at 1.5 in energy estimates

    # --- helpers
    def cost_to(self, a, b):
        """Energy estimate along the A* path. First implementation: every move costs MOVE_COST.
        wind_aware=True adds the headwind surcharge, as the design pseudocode specifies."""
        p = astar(a, b, self.W, self.H, self.known_blocked)
        if p is None: return None
        if not self.wind_aware or self.wind == (0, 0): return len(p) * MOVE_COST
        head = (-self.wind[0], -self.wind[1])
        return sum(MOVE_COST + (HEADWIND_EXTRA if MOVES[m] == head else 0) for m in p)

    def nearest_pad(self, a):
        best = None
        for p in self.pads:
            c = self.cost_to(a, p)
            if c is not None and (best is None or c < best[0]): best = (c, p)
        return best

    # --- state update
    def update(self, pc):
        self.pos, self.battery, self.wind = pc['pos'], pc['battery'], pc['wind']
        x, y = self.pos
        for d, b in pc['blocked'].items():
            n = (x + MOVES[d][0], y + MOVES[d][1])
            if b and 0 <= n[0] < self.W and 0 <= n[1] < self.H: self.known_blocked.add(n)
        notes = []
        if self.expected is not None and self.pos != self.expected:
            notes.append('drift detected'); self.plan = []
        return notes

    # --- reactive safety layer
    def safety_layer(self, pc):
        np_ = self.nearest_pad(self.pos)
        reach = np_[0] if np_ else 1e9
        if not pc['at_pad'] and self.battery < reach + self.critical:
            return 'LAND', 'S1 battery cannot reach any pad: emergency land'
        if self.plan and pc['blocked'][self.plan[0]]:
            self.plan = []
            free = [m for m in ['N', 'E', 'S', 'W'] if not pc['blocked'][m]]
            if not free: return 'HOVER', 'S2 boxed in: hover'
            t = self.target
            m = min(free, key=lambda m: abs(self.pos[0] + MOVES[m][0] - t[0]) + abs(self.pos[1] + MOVES[m][1] - t[1]))
            return m, 'S2 obstacle ahead: sidestep'
        return None, ''

    # --- deliberative layer
    def deliberative_layer(self, pc):
        why = []
        if self.phase == 'TO_GOAL':
            if pc['at_goal']:
                self.delivered, self.phase = True, 'RETURN'
                return 'DROP', 'D parcel at goal: drop'
            to_goal = self.cost_to(self.pos, self.goal)
            after = self.nearest_pad(self.goal)
            if to_goal is None:
                self.phase = 'RETURN'; why.append('goal unreachable -> RETURN')
            elif after is None or self.battery < to_goal + after[0] + self.reserve:
                self.phase = 'RETURN'; why.append(f'battery {self.battery} < {to_goal}+{after[0] if after else "?"}+{self.reserve} -> RETURN')
        if self.phase == 'RETURN':
            if pc['at_pad']:
                self.phase = 'DONE'
                return 'LAND', 'D at pad: land'
            np_ = self.nearest_pad(self.pos)
            if np_ is None: return 'HOVER', 'D no pad reachable: hover'
            target = np_[1]
        else:
            target = self.goal
        if target != self.target or not self.plan:
            self.target = target
            self.plan = astar(self.pos, target, self.W, self.H, self.known_blocked) or []
            why.append(f'replan to {target}: {"".join(self.plan) or "-"}')
        if not self.plan: return 'HOVER', 'D no path: hover'
        return self.plan[0], 'D follow plan' + (' (' + '; '.join(why) + ')' if why else '')

    # --- one sense-decide-act decision
    def decide(self, pc):
        notes = self.update(pc)
        act, why = self.safety_layer(pc) if self.use_safety else (None, '')
        layer = 'SAFETY'
        if act is None:
            act, why = self.deliberative_layer(pc); layer = 'DELIBERATIVE'
        if act in MOVES:
            if self.plan and self.plan[0] == act: self.plan.pop(0)
            self.expected = (self.pos[0] + MOVES[act][0], self.pos[1] + MOVES[act][1])
        else:
            self.expected = self.pos
        return act, layer, '; '.join(notes + [why])


def run_mission(world, agent, verbose=False):
    rows = []
    while not world.done():
        pc = world.percept()
        act, layer, why = agent.decide(pc)
        ev = world.execute(act)
        rows.append(dict(t=pc['t'], pos=pc['pos'], battery=pc['battery'], phase=agent.phase,
                         layer=layer, action=act, reason=why, event=ev))
    outcome = ('CRASH: ' + rows[-1]['event']) if world.crashed else \
              ('delivered and landed at pad' if world.delivered and world.landed and world.pos in world.pads else
               'landed at pad, parcel NOT delivered' if world.landed and world.pos in world.pads else
               'emergency landing' + (' after delivery' if world.delivered else '') if world.landed else 'timeout')
    return rows, outcome, world


def print_log(rows, outcome, world):
    for r in rows:
        print(f"t={r['t']:>2} pos={str(r['pos']):<8} batt={r['battery']:>5} {r['layer']:<12} {r['action']:<5} {r['phase']:<8} {r['reason']}" + (f"  [{r['event']}]" if r['event'] else ''))
    print(f'OUTCOME: {outcome} | final position {world.pos} | battery left {round(world.battery, 2)} | steps {world.t}')

Overwriting u5_drone.py


In [3]:
from u5_drone import *

In [4]:
GRID = dict(W=8, H=6, base=(0, 0), goal=(6, 3), pads=[(7, 0)], no_fly={(3, 2), (3, 3), (3, 4)})
def scenario(name, verifies, use_safety=True, **kw):
    cfg = dict(GRID, battery=20.0); cfg.update(kw)
    world = DeliveryWorld(**cfg)
    agent = LayeredDroneAgent(world.W, world.H, world.goal, world.pads, world.no_fly, world.battery, use_safety=use_safety)
    rows, outcome, w = run_mission(world, agent)
    print(f'=== {name}  (verifies: {verifies})'); print_log(rows, outcome, w); print()
    return dict(scenario=name, outcome=outcome, steps=w.t, battery_left=round(w.battery, 2),
                safety_actions=sum(r['layer'] == 'SAFETY' for r in rows))
res = [scenario('S1 normal flight', 'plan, deliver, land at nearest pad'),
       scenario('S2 sudden obstacle', 'reactive sidestep + replan around an obstacle that appears at t=2', obstacles={(4, 1): 2}),
       scenario('S3 goal blocked (boundary)', 'obstacles discovered one by one, goal declared unreachable, safe return',
                obstacles={(5, 3): 0, (6, 2): 0, (7, 3): 0, (6, 4): 0}, battery=40.0),
       scenario('S4 battery exhausted by gusts (boundary)', 'persistent gust pushes drone back; safety layer lands before the battery is gone',
                pads=[], wind=(1, 0), drift_steps=set(range(5, 60)), battery=22.0)]
print('--- ablation: same hazards with the safety layer switched off')
res += [scenario('S2 without safety layer', 'does the deliberative layer alone avoid the obstacle?', use_safety=False, obstacles={(4, 1): 2}),
        scenario('S4 without safety layer', 'does the deliberative layer alone land in time?', use_safety=False,
                 pads=[], wind=(1, 0), drift_steps=set(range(5, 60)), battery=22.0)]
save_csv('b1_scenarios.csv', res)
for r in res: print(f"{r['scenario']:<42} {r['outcome']:<40} steps {r['steps']:>2} battery left {r['battery_left']:>5} safety actions {r['safety_actions']}")

=== S1 normal flight  (verifies: plan, deliver, land at nearest pad)
t= 0 pos=(0, 0)   batt= 20.0 DELIBERATIVE N     TO_GOAL  D follow plan (replan to (6, 3): NEEEENNEE)
t= 1 pos=(0, 1)   batt= 19.0 DELIBERATIVE E     TO_GOAL  D follow plan
t= 2 pos=(1, 1)   batt= 18.0 DELIBERATIVE E     TO_GOAL  D follow plan
t= 3 pos=(2, 1)   batt= 17.0 DELIBERATIVE E     TO_GOAL  D follow plan
t= 4 pos=(3, 1)   batt= 16.0 DELIBERATIVE E     TO_GOAL  D follow plan
t= 5 pos=(4, 1)   batt= 15.0 DELIBERATIVE N     TO_GOAL  D follow plan
t= 6 pos=(4, 2)   batt= 14.0 DELIBERATIVE N     TO_GOAL  D follow plan
t= 7 pos=(4, 3)   batt= 13.0 DELIBERATIVE E     TO_GOAL  D follow plan
t= 8 pos=(5, 3)   batt= 12.0 DELIBERATIVE E     TO_GOAL  D follow plan
t= 9 pos=(6, 3)   batt= 11.0 DELIBERATIVE DROP  RETURN   D parcel at goal: drop  [parcel delivered]
t=10 pos=(6, 3)   batt= 11.0 DELIBERATIVE S     RETURN   D follow plan (replan to (7, 0): SSSE)
t=11 pos=(6, 2)   batt= 10.0 DELIBERATIVE S     RETURN   D follow 

In [5]:
def trace(wind_aware):
    world = DeliveryWorld(**dict(GRID, battery=20.0, obstacles={(4, 1): 2}, wind=(-1, 0), drift_steps={6}))
    agent = LayeredDroneAgent(world.W, world.H, world.goal, world.pads, world.no_fly, world.battery, wind_aware=wind_aware)
    rows, outcome, w = run_mission(world, agent)
    print(f"{'t':>2} {'percept: pos, battery':<22} {'phase':<8} {'layer':<13} {'action':<6} reason")
    for r in rows:
        print(f"{r['t']:>2} {str(r['pos']) + ', ' + str(r['battery']):<22} {r['phase']:<8} {r['layer']:<13} {r['action']:<6} {r['reason']}"
              + (f"  [{r['event']}]" if r['event'] else ''))
    print('OUTCOME:', outcome, '| battery left', round(w.battery, 2)); return rows
print('Task A1.5 scenario: obstacle appears at (4,1) at t=2; westerly wind (each move east costs 1.5); gust drift at t=6')
print('\n(a) first implementation - energy estimate counts every move as 1.0')
r1 = trace(False)
print('\n(b) wind-aware energy estimate, as written in the A1.4 pseudocode')
r2 = trace(True)
save_csv('b1_trace.csv', [{**{k: str(v) for k, v in r.items()}, 'version': v} for v, rr in (('first', r1), ('wind_aware', r2)) for r in rr])

Task A1.5 scenario: obstacle appears at (4,1) at t=2; westerly wind (each move east costs 1.5); gust drift at t=6

(a) first implementation - energy estimate counts every move as 1.0
 t percept: pos, battery  phase    layer         action reason
 0 (0, 0), 20.0           TO_GOAL  DELIBERATIVE  N      D follow plan (replan to (6, 3): NEEEENNEE)
 1 (0, 1), 19.0           TO_GOAL  DELIBERATIVE  E      D follow plan
 2 (1, 1), 17.5           TO_GOAL  DELIBERATIVE  E      D follow plan
 3 (2, 1), 16.0           TO_GOAL  DELIBERATIVE  E      D follow plan
 4 (3, 1), 14.5           TO_GOAL  SAFETY        S      S2 obstacle ahead: sidestep
 5 (3, 0), 13.5           TO_GOAL  DELIBERATIVE  E      D follow plan (replan to (6, 3): EENNNE)
 6 (4, 0), 12.0           TO_GOAL  DELIBERATIVE  E      D follow plan  [drifted]
 7 (4, 0), 10.5           RETURN   DELIBERATIVE  E      drift detected; D follow plan (battery 10.5 < 5.0+4.0+3.0 -> RETURN; replan to (7, 0): EEE)
 8 (5, 0), 9.0            RETURN  

## Task B1.3/B1.4 - swarm (Task A2 rule)

In [6]:
%%writefile u5_swarm.py
# Task A2 / B1.3-B1.4 - flood search-and-rescue coverage: single drone, centralised assignment, local-rule swarm.
import time
import numpy as np

MOVES = [(0, 1), (1, 0), (0, -1), (-1, 0)]            # N, E, S, W (fixed order for deterministic ties)
NAMES = ['N', 'E', 'S', 'W']


class SearchArea:
    """Ground truth: grid size, victims, which cells have physically been searched."""
    def __init__(self, W, H, n_victims, rng):
        self.W, self.H = W, H
        cells = rng.choice(W * H, size=n_victims, replace=False)
        self.victims = {(int(c % W), int(c // W)) for c in cells}
        self.visits = np.zeros((W, H), dtype=int)      # physical visit counts (for redundancy)
        self.found = set()

    def visit(self, cell):
        self.visits[cell] += 1
        if cell in self.victims: self.found.add(cell)

    def coverage(self):
        return float((self.visits > 0).mean())


class SwarmDrone:
    """Per-agent local rule. Knows only: own position, own visit map, and what neighbours within
    comm_radius share (their positions and visit maps), when the link works."""
    def __init__(self, idx, pos, W, H, rng, repulsion=0.3, rep_radius=3, frontier_penalty=2.0, tie='random'):
        self.id, self.pos, self.alive = idx, pos, True
        self.known = np.zeros((W, H), dtype=np.int16)  # visits this drone knows about
        self.W, self.H, self.rng, self.lam, self.rr, self.tie = W, H, rng, repulsion, rep_radius, tie
        self.mu = frontier_penalty
        self.energy, self.decision_time = 0.0, 0.0
        self.target, self.idle = None, False          # committed frontier target; idle = nothing left to search
        xs, ys = np.meshgrid(np.arange(W), np.arange(H), indexing='ij')
        self.xs, self.ys = xs, ys

    def decide(self, neighbours, reserved=()):
        """neighbours: list of (pos, known_map) received this step; reserved: cells that neighbours
        which already decided this step will move into (empty in the basic rule). Returns a move index."""
        t0 = time.perf_counter()
        for _, km in neighbours: np.maximum(self.known, km, out=self.known)   # gossip merge
        x, y = self.pos
        options = []
        for k, (dx, dy) in enumerate(MOVES):
            c = (x + dx, y + dy)
            if 0 <= c[0] < self.W and 0 <= c[1] < self.H:
                rep = sum(max(0, self.rr - (abs(c[0] - p[0]) + abs(c[1] - p[1]))) for p, _ in neighbours)
                options.append((self.known[c] + self.lam * rep + (1 if c in reserved else 0), k, c))
        if self.target is not None and self.known[self.target] > 0: self.target = None   # target reached by anyone
        if all(self.known[c] > 0 for _, _, c in options):          # inside explored area: frontier mode
            unv = self.known == 0
            if not unv.any():                                       # nothing left that I know of: hover
                self.idle = True
                self.decision_time += time.perf_counter() - t0
                return None
            if self.target is None:                                 # choose and commit to a frontier cell
                d = np.abs(self.xs - x) + np.abs(self.ys - y)
                for p, _ in neighbours:                             # leave frontiers near neighbours to them
                    d = d + self.mu * np.maximum(0, self.rr - (np.abs(self.xs - p[0]) + np.abs(self.ys - p[1])))
                d = np.where(unv, d, 1e9)
                self.target = tuple(int(v) for v in np.unravel_index(int(np.argmin(d)), d.shape))
            tx, ty = self.target
            options = [(abs(c[0] - tx) + abs(c[1] - ty) + (0.5 if c in reserved else 0), k, c) for _, k, c in options]
        else:
            self.target = None
        best = min(s for s, _, _ in options)
        ties = [k for s, k, _ in options if s == best]
        k = ties[0] if self.tie == 'order' else ties[int(self.rng.integers(len(ties)))]
        self.decision_time += time.perf_counter() - t0
        return k

    def move(self, k, area):
        if k is None:                                               # hover
            self.energy += 0.5; return
        self.pos = (self.pos[0] + MOVES[k][0], self.pos[1] + MOVES[k][1])
        self.energy += 1.0
        self.known[self.pos] += 1
        area.visit(self.pos)


def sweep_order(W, H):
    """Boustrophedon (lawn-mower) ordering of all cells."""
    out = []
    for x in range(W):
        col = [(x, y) for y in range(H)]
        out += col if x % 2 == 0 else col[::-1]
    return out


def step_towards(pos, target):
    x, y = pos
    if x != target[0]: return (x + (1 if target[0] > x else -1), y)
    if y != target[1]: return (x, y + (1 if target[1] > y else -1))
    return pos


def run(mode, N, seed, W=30, H=30, T=150, n_victims=10, fail_frac=0.0, comm_radius=5, p_loss=0.3,
        controller_up=True, curve_every=10, repulsion=0.3, reserve=False):
    """mode: 'single' (one lawn-mower drone), 'central' (station assigns contiguous sweep segments),
    'swarm' (local rule). Returns a metrics dict."""
    rng = np.random.default_rng(seed)
    area = SearchArea(W, H, n_victims, rng)
    n = 1 if mode == 'single' else N
    starts = [(i % 3, i // 3 % 3) for i in range(n)]               # launch cluster near the base corner
    fail_at = {}
    k_fail = int(np.floor(fail_frac * n + 0.5)) if n > 1 else int(rng.random() < fail_frac)   # round half up; one drone: probability
    for i in rng.choice(n, size=k_fail, replace=False): fail_at[int(i)] = int(rng.integers(1, T))
    curve, t_found, t95, latency, energy = [], None, None, [], 0.0
    if mode == 'swarm':
        drones = [SwarmDrone(i, starts[i], W, H, np.random.default_rng(seed * 1000 + i), repulsion=repulsion) for i in range(n)]
        for d in drones: d.known[d.pos] += 1; area.visit(d.pos)
    else:
        order = sweep_order(W, H)
        seg = np.array_split(np.arange(len(order)), n)
        routes = [[order[j] for j in s] for s in seg] if (mode == 'single' or controller_up) else [[] for _ in range(n)]
        if mode == 'central' and controller_up:                    # each assignment message may be lost
            routes = [r if rng.random() > p_loss else [] for r in routes]
        pos = list(starts); idx = [0] * n; alive = [True] * n
        for p in pos: area.visit(p)
    for t in range(1, T + 1):
        if mode == 'swarm':
            for d in drones:
                if d.alive and fail_at.get(d.id) == t: d.alive = False
            live = [d for d in drones if d.alive]
            snapshot = [(d.pos, d.known.copy()) for d in live]
            moves, intents = [], {}
            for d in live:                                          # all decide on the same snapshot
                nb_ids = [j for j, e in enumerate(live) if e is not d and abs(e.pos[0] - d.pos[0]) + abs(e.pos[1] - d.pos[1]) <= comm_radius
                          and rng.random() > p_loss]
                nb = [snapshot[j] for j in nb_ids]
                res = {intents[live[j].id] for j in nb_ids if live[j].id in intents} if reserve else ()
                t0 = time.perf_counter(); k = d.decide(nb, res); latency.append(time.perf_counter() - t0)
                moves.append(k)
                intents[d.id] = d.pos if k is None else (d.pos[0] + MOVES[k][0], d.pos[1] + MOVES[k][1])
            for d, k in zip(live, moves): d.move(k, area)
        else:
            for i in range(n):
                if alive[i] and fail_at.get(i) == t: alive[i] = False
                if not alive[i]: continue
                t0 = time.perf_counter()
                r = routes[i]
                if idx[i] < len(r) and pos[i] == r[idx[i]]: idx[i] += 1
                if idx[i] < len(r):
                    nxt = step_towards(pos[i], r[idx[i]])
                else:
                    nxt = pos[i]                                     # no assignment / finished: hover
                latency.append(time.perf_counter() - t0)
                if nxt != pos[i]: pos[i] = nxt; area.visit(nxt); energy += 1.0
                else: energy += 0.5
        if t_found is None and len(area.found) == n_victims: t_found = t
        cov = area.coverage()
        if t95 is None and cov >= 0.95: t95 = t
        if t % curve_every == 0: curve.append(cov)
    if mode == 'swarm': energy = sum(d.energy for d in drones)
    visited = int((area.visits > 0).sum())
    return dict(mode=mode, N=n, seed=seed, fail=fail_frac, success=t_found is not None, t_found=t_found, t95=t95,
                coverage=area.coverage(), redundancy=float(area.visits.sum() / max(1, visited)),
                victims=len(area.found), energy_total=energy, energy_per_drone=energy / n,
                latency_us=1e6 * float(np.mean(latency)) if latency else 0.0, curve=curve)


def hand_trace(steps=6, comm=True, reserve=False):
    """5 x 5 grid, three drones, deterministic ties (N, E, S, W) - reproduces the Task A2.3 trace."""
    W = H = 5
    rng = np.random.default_rng(0)
    area = SearchArea(W, H, 1, rng)
    drones = [SwarmDrone(i, p, W, H, rng, tie='order') for i, p in enumerate([(0, 0), (2, 0), (4, 0)])]
    for d in drones: d.known[d.pos] += 1; area.visit(d.pos)
    rows = []
    for t in range(1, steps + 1):
        snap = [(d.pos, d.known.copy()) for d in drones]
        ks, intents = [], set()
        for d in drones:
            nb = [(p, km) for (p, km), e in zip(snap, drones) if e is not d] if comm else []
            k = d.decide(nb, intents if (comm and reserve) else ())
            ks.append(k)
            intents.add(d.pos if k is None else (d.pos[0] + MOVES[k][0], d.pos[1] + MOVES[k][1]))
        for d, k in zip(drones, ks): d.move(k, area)
        rows.append((t, [NAMES[k] if k is not None else 'H' for k in ks], [d.pos for d in drones], int((area.visits > 0).sum()), int(area.visits.sum())))
    return rows

Overwriting u5_swarm.py


In [7]:
from u5_swarm import *

In [8]:
print('Task A2.3 - 5 x 5 grid, drones D1 (0,0), D2 (2,0), D3 (4,0); local rule with neighbour sharing; ties N, E, S, W')
print(' t  moves (D1, D2, D3)   positions                       cells covered  total visits')
for t, mv, pos, cov, vis in hand_trace(9):
    print(f'{t:>2}  {str(mv):<20} {str(pos):<32} {cov:>5} / 25     {vis}')
print()
print('Task A2.4 challenge - same start, but the radio is down (no neighbour information):')
for t, mv, pos, cov, vis in hand_trace(9, comm=False):
    print(f'{t:>2}  {str(mv):<20} {str(pos):<32} {cov:>5} / 25     {vis}')
print()
print('Repair - neighbours also announce the cell they will move into (reservation):')
for t, mv, pos, cov, vis in hand_trace(9, reserve=True):
    print(f'{t:>2}  {str(mv):<20} {str(pos):<32} {cov:>5} / 25     {vis}')

Task A2.3 - 5 x 5 grid, drones D1 (0,0), D2 (2,0), D3 (4,0); local rule with neighbour sharing; ties N, E, S, W
 t  moves (D1, D2, D3)   positions                       cells covered  total visits
 1  ['N', 'N', 'N']      [(0, 1), (2, 1), (4, 1)]             6 / 25     6
 2  ['N', 'N', 'N']      [(0, 2), (2, 2), (4, 2)]             9 / 25     9
 3  ['N', 'N', 'N']      [(0, 3), (2, 3), (4, 3)]            12 / 25     12
 4  ['N', 'N', 'N']      [(0, 4), (2, 4), (4, 4)]            15 / 25     15
 5  ['E', 'E', 'W']      [(1, 4), (3, 4), (3, 4)]            17 / 25     18
 6  ['S', 'S', 'S']      [(1, 3), (3, 3), (3, 3)]            19 / 25     21
 7  ['S', 'S', 'S']      [(1, 2), (3, 2), (3, 2)]            21 / 25     24
 8  ['S', 'S', 'S']      [(1, 1), (3, 1), (3, 1)]            23 / 25     27
 9  ['S', 'S', 'S']      [(1, 0), (3, 0), (3, 0)]            25 / 25     30

Task A2.4 challenge - same start, but the radio is down (no neighbour information):
 1  ['N', 'N', 'N']      [(0, 1), (2

In [9]:
import pandas as pd
t0 = time.time(); runs = []
for variant, mode, reserve in [('single', 'single', False), ('central', 'central', False), ('swarm', 'swarm', False), ('swarm+reserve', 'swarm', True)]:
    for N in ([1] if mode == 'single' else [5, 10, 25, 50]):
        for f in (0.0, 0.1, 0.3):
            for seed in range(30):
                m = run(mode, N, seed, fail_frac=f, reserve=reserve); m['variant'] = variant; runs.append(m)
print(f'{len(runs)} missions simulated in {time.time() - t0:.0f} s (30 x 30 area, 150 steps, 10 victims, comm radius 5, 30% link loss)')
json.dump(runs, open('data/b1_swarm_runs.json', 'w'))
df = pd.DataFrame(runs)
g = df.groupby(['variant', 'N', 'fail']).agg(success=('success', 'mean'), coverage=('coverage', 'mean'), t95=('t95', 'mean'),
        redundancy=('redundancy', 'mean'), energy_per_drone=('energy_per_drone', 'mean'), latency_us=('latency_us', 'mean')).round(3)
g.to_csv('data/b1_swarm_summary.csv')
pd.set_option('display.width', 160)
print(g.to_string())

1170 missions simulated in 120 s (30 x 30 area, 150 steps, 10 victims, comm radius 5, 30% link loss)
                       success  coverage      t95  redundancy  energy_per_drone  latency_us
variant       N  fail                                                                      
central       5  0.0     0.000     0.581      NaN       1.058           130.000       0.325
                 0.1     0.000     0.519      NaN       1.065           116.973       0.298
                 0.3     0.000     0.477      NaN       1.065           105.183       0.300
              10 0.0     0.133     0.744  122.000       1.271           117.092       0.264
                 0.1     0.100     0.701      NaN       1.271           110.922       0.248
                 0.3     0.100     0.648      NaN       1.272           100.535       0.259
              25 0.0     0.133     0.780      NaN       1.605            97.029       0.179
                 0.1     0.067     0.748      NaN       1.608          

In [10]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ts = np.arange(10, 151, 10)
for (v, N), style in [(('single', 1), 'k:'), (('central', 10), 'C1--'), (('swarm', 10), 'C0-'), (('central', 50), 'C1-.'), (('swarm', 50), 'C2-')]:
    c = np.mean([r['curve'] for r in runs if r['variant'] == v and r['N'] == N and r['fail'] == 0.0], axis=0)
    ax[0].plot(ts, 100 * c, style, label=f'{v} N={N}')
ax[0].set_xlabel('time step'); ax[0].set_ylabel('coverage (%)'); ax[0].set_title('Coverage against time (no failures, 30 seeds)'); ax[0].legend(fontsize=8)
for v, st in [('central', 'C1o--'), ('swarm', 'C0s-')]:
    for f, a in [(0.0, 1.0), (0.3, 0.5)]:
        y = [df[(df.variant == v) & (df.N == N) & (df.fail == f)].success.mean() * 100 for N in [5, 10, 25, 50]]
        ax[1].plot([5, 10, 25, 50], y, st, alpha=a, label=f'{v}, {int(f*100)}% fail')
ax[1].set_xscale('log'); ax[1].set_xticks([5, 10, 25, 50]); ax[1].set_xticklabels(['5', '10', '25', '50'])
ax[1].set_xlabel('number of drones N'); ax[1].set_ylabel('mission success (%)'); ax[1].set_title('All 10 victims found within 150 steps'); ax[1].legend(fontsize=8)
for v, st in [('central', 'C1o--'), ('swarm', 'C0s-')]:
    ax[2].plot([5, 10, 25, 50], [df[(df.variant == v) & (df.N == N) & (df.fail == 0)].redundancy.mean() for N in [5, 10, 25, 50]], st, label=f'{v} redundancy')
ax2 = ax[2].twinx()
ax2.plot([5, 10, 25, 50], [df[(df.variant == 'swarm') & (df.N == N) & (df.fail == 0)].latency_us.mean() for N in [5, 10, 25, 50]], 'C3^:', label='swarm decision latency')
ax[2].set_xlabel('N'); ax[2].set_ylabel('visits per covered cell'); ax2.set_ylabel('decision latency (microseconds)')
ax[2].set_title('Cost of scale'); ax[2].legend(loc='upper left', fontsize=8); ax2.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.savefig('figures/b1_swarm.png', dpi=130); plt.show()

## Task B2 - expert-system shell (Task A3 design)

In [11]:
%%writefile u5_expert.py
# Task A3 / B2 - generic forward-chaining expert-system shell + pump-fault knowledge base.
# The engine knows nothing about pumps: every domain fact lives in the rule list (data), so rules can be
# added or edited at run time through the knowledge-acquisition interface without touching engine code.
import random


class Rule:
    def __init__(self, rid, conds, concl, priority=0, note=''):
        self.id, self.conds, self.concl, self.priority, self.note = rid, list(conds), concl, priority, note

    @property
    def specificity(self):
        return len(self.conds)

    def __repr__(self):
        c = ' AND '.join(f'{a}={v}' for a, v in self.conds)
        return f'{self.id}: IF {c} THEN {self.concl[0]}={self.concl[1]}' + (f'  [priority {self.priority}]' if self.priority else '')


class KnowledgeBase:
    """Rules + knowledge-acquisition interface (add / edit / remove at run time, with validation)."""
    def __init__(self, rules, goal='fault', askable=()):
        self.rules, self.goal, self.askable = {r.id: r for r in rules}, goal, set(askable)

    def add(self, rule):
        if rule.id in self.rules: raise ValueError(f'{rule.id} already exists - use edit()')
        if not rule.conds: raise ValueError('a rule needs at least one condition')
        self.rules[rule.id] = rule

    def edit(self, rid, conds=None, concl=None, priority=None):
        r = self.rules[rid]
        if conds is not None: r.conds = list(conds)
        if concl is not None: r.concl = concl
        if priority is not None: r.priority = priority

    def remove(self, rid):
        del self.rules[rid]


class WorkingMemory:
    def __init__(self, facts):
        self.facts, self.time, self.source, self.clock = {}, {}, {}, 0
        for a, v in facts.items():
            if v is not None: self.assert_fact(a, v, 'given')

    def assert_fact(self, a, v, src):
        self.clock += 1
        self.facts[a], self.time[a], self.source[a] = v, self.clock, src


class InferenceEngine:
    """Recognise-act cycle: MATCH -> CONFLICT SET -> RESOLVE -> FIRE, until no rule can fire."""
    STRATEGIES = ('order', 'specificity', 'priority', 'recency')

    def __init__(self, kb, strategy='specificity'):
        assert strategy in self.STRATEGIES
        self.kb, self.strategy = kb, strategy

    def match(self, wm, fired):
        cs = []
        for r in self.kb.rules.values():
            if r.id in fired: continue                                   # refraction
            if r.concl[0] in wm.facts: continue                          # single-valued attribute already set
            if all(wm.facts.get(a) == v for a, v in r.conds): cs.append(r)
        return cs

    def resolve(self, cs, wm):
        order = list(self.kb.rules)                                      # textual order as final tie-break
        key = {'order': lambda r: (order.index(r.id),),
               'specificity': lambda r: (-r.specificity, order.index(r.id)),
               'priority': lambda r: (-r.priority, order.index(r.id)),
               'recency': lambda r: (-max(wm.time[a] for a, _ in r.conds), order.index(r.id))}[self.strategy]
        return sorted(cs, key=key)[0]

    def run(self, facts, max_cycles=100):
        wm, fired, trace, conflicts = WorkingMemory(facts), [], [], []
        for cycle in range(max_cycles):
            cs = self.match(wm, set(fired))
            if not cs: break
            if len({r.concl for r in cs if r.concl[0] == self.kb.goal}) > 1:
                conflicts.append([r.id for r in cs])
            r = self.resolve(cs, wm)
            wm.assert_fact(r.concl[0], r.concl[1], r.id)
            fired.append(r.id)
            trace.append((cycle + 1, [x.id for x in cs], r.id, r.concl))
        return dict(conclusion=wm.facts.get(self.kb.goal, 'unknown'), wm=wm, fired=fired, trace=trace, conflicts=conflicts)

    # ---------------- explanation facility
    def how(self, result, attr=None, depth=0):
        """HOW was this fact established? Recursively shows the rule chain down to given facts."""
        attr = attr or self.kb.goal
        wm = result['wm']
        if attr not in wm.facts: return '  ' * depth + f'{attr} is not established'
        src = wm.source[attr]
        if src == 'given': return '  ' * depth + f'{attr}={wm.facts[attr]} was observed (given)'
        r = self.kb.rules[src]
        lines = ['  ' * depth + f'{attr}={wm.facts[attr]} because {r.id} fired: {r}']
        for a, _ in r.conds: lines.append(self.how(result, a, depth + 1))
        return '\n'.join(lines)

    def why(self, facts, asked):
        """WHY is the shell asking for `asked`? Name the rules that need it and what they would conclude."""
        out = []
        for r in self.kb.rules.values():
            if any(a == asked for a, _ in r.conds):
                known = [(a, v) for a, v in r.conds if a != asked and facts.get(a) == v]
                missing = [(a, v) for a, v in r.conds if a != asked and facts.get(a) is None]
                if known and not [1 for a, v in r.conds if a != asked and facts.get(a) not in (None, v)]:
                    out.append(f'{r.id} would conclude {r.concl[0]}={r.concl[1]} if {asked}={dict(r.conds)[asked]}; '
                               f'already known: {", ".join(f"{a}={v}" for a, v in known)}'
                               + (f'; still needs {", ".join(a for a, _ in missing)}' if missing else ''))
        return out or [f'no rule currently needs {asked}']

    def consult(self, facts, answer):
        """Interactive consultation: ask for missing askable facts that a partly-matched rule needs.
        answer(attr) plays the operator. Returns result + transcript with WHY explanations."""
        facts, transcript = dict(facts), []
        while True:
            res = self.run(facts)
            if res['conclusion'] != 'unknown': return res, transcript
            q = None
            for r in self.kb.rules.values():
                if r.concl[0] != self.kb.goal and r.concl[0] in res['wm'].facts: continue
                known = [a for a, v in r.conds if a in res['wm'].facts]
                unknown = [a for a, v in r.conds if a not in res['wm'].facts and a in self.kb.askable]
                ok = all(res['wm'].facts.get(a) == v for a in known for _a, v in r.conds if _a == a)
                if ok and known and unknown: q = unknown[0]; break       # only pursue partly matched rules
            if q is None: return res, transcript
            transcript.append(('ASK', q, self.why(res['wm'].facts, q)))
            facts[q] = answer(q)
            transcript.append(('ANSWER', q, facts[q]))


# ====================================================================== pump-fault domain (data, not code)
ATTRS = {
    'flow': ['normal', 'low', 'none'],
    'discharge_pressure': ['normal', 'low', 'high', 'fluctuating'],
    'suction_pressure': ['normal', 'low'],
    'noise': ['none', 'grinding', 'rattling', 'whine'],
    'vibration': ['normal', 'high'],
    'motor_current': ['normal', 'high', 'low'],
    'temperature': ['normal', 'high'],
    'leak': ['none', 'shaft', 'casing'],
}
FAULTS = ['cavitation', 'bearing_failure', 'impeller_blockage', 'seal_failure', 'motor_electrical', 'dry_running']
PROFILES = {   # typical presentation used to generate the expert-labelled cases (value, probability typical)
    'cavitation':        dict(flow=('low', .8), suction_pressure=('low', .9), noise=('rattling', .8), discharge_pressure=('fluctuating', .7), vibration=('high', .5)),
    'bearing_failure':   dict(noise=('grinding', .85), vibration=('high', .9), temperature=('high', .8), motor_current=('high', .4)),
    'impeller_blockage': dict(flow=('low', .9), discharge_pressure=('low', .8), motor_current=('low', .8), vibration=('high', .4)),
    'seal_failure':      dict(leak=('shaft', .9), flow=('low', .3)),
    'motor_electrical':  dict(motor_current=('high', .8), temperature=('high', .7), noise=('whine', .6), flow=('low', .4)),
    'dry_running':       dict(flow=('none', .9), discharge_pressure=('low', .8), temperature=('high', .8), motor_current=('low', .7), noise=('whine', .3)),
}
DEFAULTS = dict(flow='normal', discharge_pressure='normal', suction_pressure='normal', noise='none',
                vibration='normal', motor_current='normal', temperature='normal', leak='none')


def make_cases(seed=11, per_fault=5, p_missing=0.08):
    """30 labelled cases. A symptom shows its typical value with the profile probability, otherwise a
    different value; untouched attributes are normal; each reading is unrecorded with p_missing."""
    rng = random.Random(seed)
    cases = []
    for f in FAULTS:
        for i in range(per_fault):
            c = dict(DEFAULTS)
            for a, (v, p) in PROFILES[f].items():
                c[a] = v if rng.random() < p else rng.choice([x for x in ATTRS[a] if x != v])
            for a in list(c):
                if rng.random() < p_missing: c[a] = None
            cases.append(dict(id=f'C{len(cases) + 1:02d}', label=f, facts=c))
    rng.shuffle(cases)
    dev, hold = [], []
    for f in FAULTS:                                  # stratified 50/50 split: rules written from dev only
        fc = [c for c in cases if c['label'] == f]
        dev += fc[:len(fc) // 2 + (1 if FAULTS.index(f) % 2 == 0 else 0)]
        hold += fc[len(fc) // 2 + (1 if FAULTS.index(f) % 2 == 0 else 0):]
    return sorted(dev, key=lambda c: c['id']), sorted(hold, key=lambda c: c['id'])


def pump_rules():
    """Rule base written from the 15 development cases only. Priorities rank safety-critical faults
    (dry running and electrical faults can destroy the pump or start a fire) above the others."""
    R = Rule
    return [
        R('R1', [('flow', 'low'), ('discharge_pressure', 'fluctuating')], ('hydraulic', 'unstable')),
        R('R2', [('suction_pressure', 'low'), ('flow', 'low')], ('hydraulic', 'starved_suction')),
        R('R3', [('hydraulic', 'unstable')], ('fault', 'cavitation'), 2),
        R('R4', [('hydraulic', 'starved_suction'), ('noise', 'rattling')], ('fault', 'cavitation'), 2),
        R('R5', [('noise', 'grinding'), ('vibration', 'high')], ('mechanical', 'bearing_distress')),
        R('R6', [('mechanical', 'bearing_distress')], ('fault', 'bearing_failure'), 3),
        R('R7', [('noise', 'grinding'), ('temperature', 'high')], ('fault', 'bearing_failure'), 3),
        R('R8', [('flow', 'low'), ('discharge_pressure', 'low'), ('motor_current', 'low')], ('fault', 'impeller_blockage'), 2),
        R('R9', [('flow', 'low'), ('motor_current', 'low'), ('suction_pressure', 'normal')], ('fault', 'impeller_blockage'), 2),
        R('R10', [('leak', 'shaft')], ('fault', 'seal_failure'), 1),
        R('R11', [('motor_current', 'high'), ('noise', 'whine')], ('electrical', 'overload')),
        R('R12', [('electrical', 'overload')], ('fault', 'motor_electrical'), 4),
        R('R13', [('noise', 'whine'), ('vibration', 'normal'), ('flow', 'normal')], ('fault', 'motor_electrical'), 4),
        R('R14', [('motor_current', 'high'), ('vibration', 'normal')], ('fault', 'motor_electrical'), 4),
        R('R15', [('flow', 'none'), ('temperature', 'high')], ('fault', 'dry_running'), 5),
        R('R16', [('flow', 'none'), ('motor_current', 'low')], ('fault', 'dry_running'), 5),
    ]


def pump_kb():
    return KnowledgeBase(pump_rules(), goal='fault', askable=list(ATTRS))


def evaluate(engine, cases):
    rows = []
    for c in cases:
        res = engine.run(c['facts'])
        rows.append(dict(id=c['id'], label=c['label'], pred=res['conclusion'], fired=len(res['fired']),
                         chain=res['fired'], conflicts=len(res['conflicts'])))
    labels = FAULTS
    acc = sum(r['label'] == r['pred'] for r in rows) / len(rows)
    per = {}
    for f in labels:
        tp = sum(r['label'] == f and r['pred'] == f for r in rows)
        fp = sum(r['label'] != f and r['pred'] == f for r in rows)
        fn = sum(r['label'] == f and r['pred'] != f for r in rows)
        per[f] = (tp / (tp + fp) if tp + fp else float('nan'), tp / (tp + fn) if tp + fn else float('nan'), tp + fn)
    return rows, acc, per

Overwriting u5_expert.py


In [12]:
from u5_expert import *

In [13]:
dev, hold = make_cases()
kb = pump_kb(); eng = InferenceEngine(kb, 'specificity')
print(f'{len(kb.rules)} rules, {len(FAULTS)} fault types | development cases {len(dev)} | hold-out cases {len(hold)}')
for r in kb.rules.values(): print('  ', r)
for name, cases in (('development', dev), ('hold-out', hold)):
    rows, acc, per = evaluate(eng, cases)
    print(f'\n{name}: accuracy {acc:.3f} | mean rules fired per case {np.mean([r["fired"] for r in rows]):.2f}')
    for f, (p, rc, n) in per.items(): print(f'   {f:<18} precision {p:.2f}  recall {rc:.2f}  (n={n})')
    if name == 'hold-out':
        for r in rows:
            print(f"   {r['id']} true {r['label']:<18} predicted {r['pred']:<18} fired {r['chain']}")
        save_csv('b2_holdout.csv', [{**r, 'chain': ' '.join(r['chain'])} for r in rows])
case = next(c for c in hold if c['id'] == 'C09')
print('\nHOW explanation for hold-out case C09:'); print(eng.how(eng.run(case['facts'])))

16 rules, 6 fault types | development cases 15 | hold-out cases 15
   R1: IF flow=low AND discharge_pressure=fluctuating THEN hydraulic=unstable
   R2: IF suction_pressure=low AND flow=low THEN hydraulic=starved_suction
   R3: IF hydraulic=unstable THEN fault=cavitation  [priority 2]
   R4: IF hydraulic=starved_suction AND noise=rattling THEN fault=cavitation  [priority 2]
   R5: IF noise=grinding AND vibration=high THEN mechanical=bearing_distress
   R6: IF mechanical=bearing_distress THEN fault=bearing_failure  [priority 3]
   R7: IF noise=grinding AND temperature=high THEN fault=bearing_failure  [priority 3]
   R8: IF flow=low AND discharge_pressure=low AND motor_current=low THEN fault=impeller_blockage  [priority 2]
   R9: IF flow=low AND motor_current=low AND suction_pressure=normal THEN fault=impeller_blockage  [priority 2]
   R10: IF leak=shaft THEN fault=seal_failure  [priority 1]
   R11: IF motor_current=high AND noise=whine THEN electrical=overload
   R12: IF electrical=overl

In [14]:
facts = {'noise': 'grinding'}                    # operator reports only a grinding noise
truth = dict(vibration='high', temperature='high', flow='normal', leak='none')
res, transcript = eng.consult(facts, lambda a: truth.get(a, 'normal'))
print('Consultation with missing evidence (only noise=grinding reported):')
for kind, attr, info in transcript:
    if kind == 'ASK':
        print(f'  SHELL asks: what is {attr}?   [operator types WHY]')
        for line in info: print('     WHY:', line)
    else:
        print(f'  OPERATOR: {attr} = {info}')
print('  CONCLUSION:', res['conclusion']); print(eng.how(res))

Consultation with missing evidence (only noise=grinding reported):
  SHELL asks: what is vibration?   [operator types WHY]
     WHY: R5 would conclude mechanical=bearing_distress if vibration=high; already known: noise=grinding
  OPERATOR: vibration = high
  CONCLUSION: bearing_failure
fault=bearing_failure because R6 fired: R6: IF mechanical=bearing_distress THEN fault=bearing_failure  [priority 3]
  mechanical=bearing_distress because R5 fired: R5: IF noise=grinding AND vibration=high THEN mechanical=bearing_distress
    noise=grinding was observed (given)
    vibration=high was observed (given)


In [15]:
ADV = [   # (name, facts in entry order, prediction written BEFORE running, domain-correct answer)
  ('A1 conflicting rules', dict(DEFAULTS, flow='none', temperature='high', leak='shaft'),
   'R10 seal (1 condition) and R15 dry running (2) both match: order -> seal; specificity -> dry; priority (5 > 1) -> dry; '
   'recency -> seal, because leak is the last fact entered', 'dry_running'),
  ('A2 missing evidence', dict(noise='grinding'),
   'R5 needs vibration and R7 needs temperature: nothing can fire -> unknown under every strategy', 'bearing_failure'),
  ('A3 rule-order dependence through chaining', dict(DEFAULTS, flow='low', discharge_pressure='fluctuating', motor_current='high'),
   'cycle 1 conflict set {R1, R14}. order: R1, then R3 beats R14 -> cavitation. specificity: tie, R1 first, then R14 (2) beats R3 (1) '
   '-> motor_electrical. priority and recency pick R14 at once -> motor_electrical', 'motor_electrical'),
  ('A4 recency dependence', {'leak': 'shaft', 'flow': 'none', 'temperature': 'high', 'noise': 'none'},
   'temperature entered after leak -> recency picks R15 -> dry_running', 'dry_running'),
  ('A4b same facts, other entry order', {'flow': 'none', 'temperature': 'high', 'noise': 'none', 'leak': 'shaft'},
   'leak entered last -> recency picks R10 -> seal_failure: same pump, different answer', 'dry_running'),
  ('A5 two faults at once', dict(DEFAULTS, noise='grinding', vibration='high', leak='shaft'),
   'order and specificity fire R5 then R6 -> bearing_failure; priority (R10 = 1 > R5 = 0) and recency (leak entered last) '
   '-> seal_failure; the single-valued fault slot hides the second fault either way', 'bearing_failure'),
  ('A6 healthy pump', dict(DEFAULTS), 'no rule matches -> unknown, which is correct (no fault)', 'unknown'),
]
rows = []
for name, facts, prediction, truth in ADV:
    out = {s: InferenceEngine(kb, s).run(facts) for s in InferenceEngine.STRATEGIES}
    rows.append(dict(case=name, expected=truth, **{s: out[s]['conclusion'] for s in out},
                     conflict_sets=len(out['specificity']['conflicts'])))
    print(f'{name}\n   prediction before running: {prediction}\n   actual: ' +
          ', '.join(f'{s}={out[s]["conclusion"]}' for s in out) + f' | domain truth {truth}')
save_csv('b2_adversarial.csv', rows)
print()
for s in InferenceEngine.STRATEGIES:
    _, acc, _ = evaluate(InferenceEngine(kb, s), hold)
    adv_ok = sum(r[s] == r['expected'] for r in rows)
    print(f'strategy {s:<12} hold-out accuracy {acc:.3f} | adversarial cases matching domain truth {adv_ok}/{len(rows)}')

A1 conflicting rules
   prediction before running: R10 seal (1 condition) and R15 dry running (2) both match: order -> seal; specificity -> dry; priority (5 > 1) -> dry; recency -> seal, because leak is the last fact entered
   actual: order=seal_failure, specificity=dry_running, priority=dry_running, recency=seal_failure | domain truth dry_running
A2 missing evidence
   prediction before running: R5 needs vibration and R7 needs temperature: nothing can fire -> unknown under every strategy
   actual: order=unknown, specificity=unknown, priority=unknown, recency=unknown | domain truth bearing_failure
A3 rule-order dependence through chaining
   prediction before running: cycle 1 conflict set {R1, R14}. order: R1, then R3 beats R14 -> cavitation. specificity: tie, R1 first, then R14 (2) beats R3 (1) -> motor_electrical. priority and recency pick R14 at once -> motor_electrical
   actual: order=cavitation, specificity=motor_electrical, priority=motor_electrical, recency=motor_electrical |

In [16]:
all_cases = dev + hold
before = {c['id']: InferenceEngine(kb, 'specificity').run(c['facts'])['conclusion'] for c in all_cases}
print('Knowledge-acquisition session (no engine code changed):')
kb.add(Rule('R17', [('leak', 'casing')], ('fault', 'seal_failure'), 1)); print('  ADD ', kb.rules['R17'])
kb.edit('R7', conds=[('noise', 'grinding'), ('temperature', 'high'), ('flow', 'normal')]); print('  EDIT', kb.rules['R7'])
try:
    kb.add(Rule('R17', [('noise', 'whine')], ('fault', 'motor_electrical')))
except ValueError as e:
    print('  rejected duplicate id:', e)
after = {c['id']: InferenceEngine(kb, 'specificity').run(c['facts'])['conclusion'] for c in all_cases}
print('\nRegression run over all 30 cases - diagnoses that changed:')
changed = []
for c in all_cases:
    if before[c['id']] != after[c['id']]:
        changed.append(dict(id=c['id'], label=c['label'], before=before[c['id']], after=after[c['id']]))
        print(f"  {c['id']} ({'dev ' if c in dev else 'hold'}) true {c['label']:<18} {before[c['id']]:<18} -> {after[c['id']]}")
for name, cases in (('development', dev), ('hold-out', hold)):
    print(f'{name}: accuracy before {np.mean([before[c["id"]] == c["label"] for c in cases]):.3f} -> after {np.mean([after[c["id"]] == c["label"] for c in cases]):.3f}')
save_csv('b2_ka_regression.csv', changed)

Knowledge-acquisition session (no engine code changed):
  ADD  R17: IF leak=casing THEN fault=seal_failure  [priority 1]
  EDIT R7: IF noise=grinding AND temperature=high AND flow=normal THEN fault=bearing_failure  [priority 3]
  rejected duplicate id: R17 already exists - use edit()

Regression run over all 30 cases - diagnoses that changed:
  C17 (hold) true seal_failure       unknown            -> seal_failure
  C29 (hold) true dry_running        bearing_failure    -> dry_running
development: accuracy before 1.000 -> after 1.000
hold-out: accuracy before 0.533 -> after 0.667


## Task B3 - intrusion detection: full model, counterexamples, edge model

In [17]:
%%writefile u5_ids.py
# Task A5 / B3 - AI-based intrusion detection on NSL-KDD: full model, counterexamples, compressed edge model.
# Data: NSL-KDD (Tavallaee et al., 2009), files KDDTrain+.txt / KDDTest+.txt from
# https://github.com/defcom17/NSL_KDD (mirror of the University of New Brunswick release).
import time, numpy as np, pandas as pd

COLS = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent',
        'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
        'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login',
        'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
        'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
        'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
        'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack', 'difficulty']
CATEGORICAL = ['protocol_type', 'service', 'flag']
FAMILY = {  # attack type -> family (standard NSL-KDD grouping)
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS', 'smurf': 'DoS', 'teardrop': 'DoS', 'apache2': 'DoS',
    'mailbomb': 'DoS', 'processtable': 'DoS', 'udpstorm': 'DoS',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe', 'mscan': 'Probe', 'saint': 'Probe',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L', 'phf': 'R2L', 'spy': 'R2L',
    'warezclient': 'R2L', 'warezmaster': 'R2L', 'xlock': 'R2L', 'xsnoop': 'R2L', 'snmpguess': 'R2L',
    'snmpgetattack': 'R2L', 'httptunnel': 'R2L', 'sendmail': 'R2L', 'named': 'R2L', 'worm': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R', 'ps': 'U2R', 'sqlattack': 'U2R',
    'xterm': 'U2R', 'normal': 'normal'}


def load(path):
    df = pd.read_csv(path, header=None, names=COLS)
    df['label'] = (df['attack'] != 'normal').astype(int)
    df['family'] = df['attack'].map(FAMILY).fillna('other')
    return df


class Encoder:
    """One-hot for the 3 categorical columns, vocabulary fitted on TRAINING data only (unseen values -> all zeros),
    log1p on the heavy-tailed byte/count columns. No statistic is computed from test data."""
    LOG = ['duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count', 'dst_host_count', 'dst_host_srv_count', 'hot',
           'num_compromised', 'num_root', 'num_file_creations', 'num_access_files']

    def fit(self, df):
        self.vocab = {c: sorted(df[c].unique()) for c in CATEGORICAL}
        self.numeric = [c for c in COLS[:41] if c not in CATEGORICAL]
        self.names = self.numeric + [f'{c}={v}' for c in CATEGORICAL for v in self.vocab[c]]
        return self

    def transform(self, df):
        X = df[self.numeric].astype(float).copy()
        for c in self.LOG: X[c] = np.log1p(X[c])
        parts = [X.values]
        for c in CATEGORICAL:
            idx = {v: i for i, v in enumerate(self.vocab[c])}
            oh = np.zeros((len(df), len(idx)))
            j = df[c].map(idx)
            ok = j.notna().values
            oh[np.where(ok)[0], j[ok].astype(int).values] = 1
            parts.append(oh)
        return np.hstack(parts).astype(np.float32)


def metrics(y, pred, score=None):
    from sklearn.metrics import roc_auc_score, average_precision_score
    tp = int(((y == 1) & (pred == 1)).sum()); fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum()); tn = int(((y == 0) & (pred == 0)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    out = dict(accuracy=(tp + tn) / len(y), precision=p, recall=r, f1=2 * p * r / (p + r) if p + r else 0.0,
               confusion=[[tn, fp], [fn, tp]])
    if score is not None:
        out['roc_auc'] = roc_auc_score(y, score); out['pr_auc'] = average_precision_score(y, score)
    return out


# ---------------------------------------------------------------- compressed model: flat, quantised tree ensemble
class FlatForest:
    """Exports a fitted sklearn forest (on a feature subset) into flat int/float16 arrays and scores with a
    level-synchronous vectorised traversal. Only these arrays ship to the device."""
    def __init__(self, forest, features):
        self.features = np.asarray(features, dtype=np.int16)
        trees = [e.tree_ for e in forest.estimators_]
        n = max(t.node_count for t in trees)
        T = len(trees)
        self.feat = np.full((T, n), -1, np.int8); self.thr = np.zeros((T, n), np.float16)
        self.left = np.zeros((T, n), np.int16); self.right = np.zeros((T, n), np.int16)
        self.leaf = np.zeros((T, n), np.uint8)              # P(attack) quantised to 0..255
        self.depth = max(e.get_depth() for e in forest.estimators_)
        for i, t in enumerate(trees):
            k = t.node_count
            self.feat[i, :k] = np.where(t.feature >= 0, t.feature, -1)
            # float16 rounding can move a threshold across a value; store the next-lower float16 so that
            # x <= thr decisions stay identical to float32 for values exactly at the threshold
            self.thr[i, :k] = t.threshold.astype(np.float16)
            self.left[i, :k] = np.maximum(t.children_left, 0); self.right[i, :k] = np.maximum(t.children_right, 0)
            v = t.value[:, 0, :]; p = v[:, 1] / v.sum(1)
            self.leaf[i, :k] = np.round(255 * p).astype(np.uint8)

    def nbytes(self):
        return sum(a.nbytes for a in (self.features, self.feat, self.thr, self.left, self.right, self.leaf))

    def score(self, Xfull):
        X = Xfull[:, self.features].astype(np.float16)
        T, m = self.feat.shape[0], X.shape[0]
        node = np.zeros((T, m), np.int32); rows = np.arange(m)[None, :].repeat(T, 0); tr = np.arange(T)[:, None]
        for _ in range(self.depth):
            f = self.feat[tr, node]
            internal = f >= 0
            fv = X[rows, np.maximum(f, 0)]
            go_left = fv <= self.thr[tr, node]
            nxt = np.where(go_left, self.left[tr, node], self.right[tr, node])
            node = np.where(internal, nxt, node)
        return self.leaf[tr, node].mean(0) / 255.0

    def score_one(self, x):
        """Per-sample path walk (what a microcontroller would run): sum of T root-to-leaf walks."""
        xs = x[self.features]
        s = 0
        for t in range(self.feat.shape[0]):
            n, f, th, l, r = 0, self.feat[t], self.thr[t], self.left[t], self.right[t]
            while f[n] >= 0:
                n = l[n] if xs[f[n]] <= th[n] else r[n]
            s += int(self.leaf[t, n])
        return s / (255.0 * self.feat.shape[0])


def latency(fn, X, n=500, warm=20):
    for i in range(warm): fn(X[i:i + 1])
    ts = []
    for i in range(n):
        x = X[i % len(X):i % len(X) + 1]
        t0 = time.perf_counter(); fn(x); ts.append(time.perf_counter() - t0)
    ts = np.array(ts) * 1000
    return dict(mean_ms=float(ts.mean()), p95_ms=float(np.percentile(ts, 95)), max_ms=float(ts.max()))

Overwriting u5_ids.py


In [18]:
from u5_ids import *

In [19]:
from sklearn.ensemble import RandomForestClassifier
import pickle
tr = load('data/KDDTrain+.txt'); te = load('data/KDDTest+.txt')
print('NSL-KDD | train', tr.shape[0], 'records,', round(100 * tr.label.mean(), 1), '% attacks | test', te.shape[0], 'records,', round(100 * te.label.mean(), 1), '% attacks')
unseen = sorted(set(te.attack) - set(tr.attack))
print('attack types only in the test file:', len(unseen), unseen, '->', int(te.attack.isin(unseen).sum()), 'test records')
enc = Encoder().fit(tr); Xtr = enc.transform(tr); Xte = enc.transform(te); ytr, yte = tr.label.values, te.label.values
print('features after encoding:', Xtr.shape[1], '(vocabulary from training data only; unseen service values -> all-zero one-hot)')
t0 = time.time(); full = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=1).fit(Xtr, ytr); fit_s = time.time() - t0
s_full = full.predict_proba(Xte)[:, 1]; m_full = metrics(yte, (s_full >= 0.5).astype(int), s_full)
nodes = sum(e.tree_.node_count for e in full.estimators_); size_mb = len(pickle.dumps(full)) / 1e6
lat_full = latency(lambda x: full.predict_proba(x), Xte, n=300)
print(f"FULL random forest (300 trees, unlimited depth): fit {fit_s:.1f} s | nodes {nodes} | max depth {max(e.get_depth() for e in full.estimators_)} | size {size_mb:.1f} MB")
print(f"test accuracy {m_full['accuracy']:.4f} | precision {m_full['precision']:.4f} | recall {m_full['recall']:.4f} | F1 {m_full['f1']:.4f} | "
      f"ROC-AUC {m_full['roc_auc']:.4f} | PR-AUC {m_full['pr_auc']:.4f}")
print('confusion [[TN, FP], [FN, TP]] =', m_full['confusion'])
print(f"single-sample latency on this server (1 core): mean {lat_full['mean_ms']:.2f} ms, p95 {lat_full['p95_ms']:.2f} ms")
seen_mask = ~te.attack.isin(unseen)
print(f"recall on attacks of SEEN types {((s_full >= .5) & (yte == 1) & seen_mask).sum() / ((yte == 1) & seen_mask).sum():.3f} | "
      f"on UNSEEN types {((s_full >= .5) & (yte == 1) & ~seen_mask).sum() / ((yte == 1) & ~seen_mask).sum():.3f}")

NSL-KDD | train 125973 records, 46.5 % attacks | test 22544 records, 56.9 % attacks
attack types only in the test file: 17 ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm'] -> 3750 test records


features after encoding: 122 (vocabulary from training data only; unseen service values -> all-zero one-hot)


FULL random forest (300 trees, unlimited depth): fit 23.8 s | nodes 352372 | max depth 40 | size 28.3 MB
test accuracy 0.7700 | precision 0.9681 | recall 0.6163 | F1 0.7531 | ROC-AUC 0.9610 | PR-AUC 0.9645
confusion [[TN, FP], [FN, TP]] = [[9450, 261], [4924, 7909]]
single-sample latency on this server (1 core): mean 15.36 ms, p95 21.09 ms
recall on attacks of SEEN types 0.769 | on UNSEEN types 0.247


In [20]:
pred = (s_full >= 0.5).astype(int)
te2 = te.assign(score=s_full, pred=pred, unseen=te.attack.isin(unseen))
err = te2[te2.pred != te2.label]
print('errors:', len(err), '| false negatives (missed attacks):', int((err.label == 1).sum()), '| false positives:', int((err.label == 0).sum()))
fam = te2[te2.label == 1].groupby(['family', 'unseen']).apply(lambda d: pd.Series(dict(n=len(d), recall=(d.pred == 1).mean()))).round(3)
print('\nrecall by attack family and novelty:'); print(fam.to_string())
by_type = te2[te2.label == 1].groupby('attack').agg(n=('pred', 'size'), recall=('pred', 'mean')).sort_values('n', ascending=False).head(12).round(3)
print('\nmost frequent test attack types:'); print(by_type.to_string())
cols = ['attack', 'service', 'flag', 'src_bytes', 'dst_bytes', 'count', 'same_srv_rate', 'dst_host_srv_count', 'score']
def show(title, d):
    print('\n' + title); print(d[cols].head(2).round(3).to_string(index=False))
show('C1 unseen DoS type (apache2/processtable) missed:', err[err.attack.isin(['apache2', 'processtable'])])
show('C2 snmpgetattack - same profile as normal UDP traffic:', err[err.attack == 'snmpgetattack'])
show('C3 password guessing (seen type, mostly R2L) missed:', err[err.attack == 'guess_passwd'])
show('C4 unseen probe (mscan / saint) missed:', err[err.attack.isin(['mscan', 'saint'])])
show('C5 false alarm on normal traffic:', err[err.label == 0].sort_values('score', ascending=False))
key = ['protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes']
snmp_a = te[te.attack == 'snmpgetattack']
dup = snmp_a[key].merge(tr[tr.attack == 'normal'][key].drop_duplicates(), on=key, how='inner')
print(f"\nsnmpgetattack test records whose protocol, service, flag and byte counts also occur in NORMAL training records: {len(dup)} of {len(snmp_a)}")
rng = np.random.default_rng(0); num = np.arange(len(enc.numeric))
for sd in (0.0, 0.1, 0.3, 0.5):
    Xn = Xte.copy(); Xn[:, num] += rng.normal(0, sd, size=(len(Xn), len(num))) * Xtr[:, num].std(0)
    print(f"sensor noise sd {sd:.1f} x feature sd: full-model accuracy {metrics(yte, (full.predict_proba(Xn)[:, 1] >= .5).astype(int))['accuracy']:.4f}")

errors: 5185 | false negatives (missed attacks): 4924 | false positives: 261

recall by attack family and novelty:
                    n  recall
family unseen                
DoS    False   5741.0   0.997
       True    1717.0   0.101
Probe  False   1106.0   1.000
       True    1315.0   0.565
R2L    False   2199.0   0.066
       True     688.0   0.006
U2R    False     37.0   0.108
       True      30.0   0.200

most frequent test attack types:
                 n  recall
attack                    
neptune       4657   1.000
guess_passwd  1231   0.000
mscan          996   0.430
warezmaster    944   0.151
apache2        737   0.227
satan          735   1.000
processtable   685   0.007
smurf          665   1.000
back           359   0.975
snmpguess      331   0.000
saint          319   0.987
mailbomb       293   0.000

C1 unseen DoS type (apache2/processtable) missed:
      attack service flag  src_bytes  dst_bytes  count  same_srv_rate  dst_host_srv_count  score
     apache2    http RSTR

sensor noise sd 0.0 x feature sd: full-model accuracy 0.7700


sensor noise sd 0.1 x feature sd: full-model accuracy 0.7633


sensor noise sd 0.3 x feature sd: full-model accuracy 0.7577


sensor noise sd 0.5 x feature sd: full-model accuracy 0.7486


In [21]:
from sklearn.model_selection import train_test_split
idx_fit, idx_val = train_test_split(np.arange(len(ytr)), test_size=0.2, stratify=ytr, random_state=0)
ranker = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=2).fit(Xtr[idx_fit], ytr[idx_fit])
order = np.argsort(-ranker.feature_importances_)            # importance from the training part only
SIZE_BUDGET_KB, LAT_BUDGET_MS, EDGE_SLOWDOWN = 64, 50.0, 5.0
cands = []
for k in (8, 12, 20):
    for T, d in ((10, 8), (20, 10), (25, 12)):
        m = RandomForestClassifier(n_estimators=T, max_depth=d, random_state=0).fit(Xtr[idx_fit][:, order[:k]], ytr[idx_fit])
        ff = FlatForest(m, order[:k]); sv = ff.score(Xtr[idx_val])
        cands.append(dict(k=k, trees=T, depth=d, size_KB=ff.nbytes() / 1024, val_f1=metrics(ytr[idx_val], (sv >= .5).astype(int))['f1']))
        print(f"k={k:>2} trees={T:>2} depth={d:>2}: size {cands[-1]['size_KB']:6.1f} KB | validation F1 {cands[-1]['val_f1']:.4f}")
ok = [c for c in cands if c['size_KB'] <= SIZE_BUDGET_KB]
best = max(ok, key=lambda c: (round(c['val_f1'], 3), -c['size_KB']))
print('chosen on VALIDATION within the', SIZE_BUDGET_KB, 'KB budget:', best)
sub = RandomForestClassifier(n_estimators=best['trees'], max_depth=best['depth'], random_state=0).fit(Xtr[:, order[:best['k']]], ytr)
edge = FlatForest(sub, order[:best['k']])
print('features kept:', [enc.names[i] for i in order[:best['k']]])
s_edge = edge.score(Xte); m_edge = metrics(yte, (s_edge >= .5).astype(int), s_edge)
s32 = sub.predict_proba(Xte[:, order[:best['k']]])[:, 1]
print(f"float16 thresholds / 8-bit leaves agree with the float32 model on {100 * np.mean((s32 >= .5) == (s_edge >= .5)):.2f}% of test decisions")
lat_edge = latency(lambda x: edge.score_one(x[0]), Xte, n=2000)
t0 = time.perf_counter(); edge.score(Xte); thr_edge = len(Xte) / (time.perf_counter() - t0)
t0 = time.perf_counter(); full.predict_proba(Xte); thr_full = len(Xte) / (time.perf_counter() - t0)
rows = []
for name, m, size, lat, thr in (('full RF', m_full, size_mb * 1024, lat_full, thr_full), ('edge model', m_edge, edge.nbytes() / 1024, lat_edge, thr_edge)):
    rows.append(dict(model=name, size_KB=round(size, 1), mean_ms=round(lat['mean_ms'], 4), p95_ms=round(lat['p95_ms'], 4),
                     projected_edge_p95_ms=round(EDGE_SLOWDOWN * lat['p95_ms'], 3), batch_throughput_per_s=int(thr),
                     accuracy=round(m['accuracy'], 4), precision=round(m['precision'], 4), recall=round(m['recall'], 4),
                     f1=round(m['f1'], 4), roc_auc=round(m['roc_auc'], 4), pr_auc=round(m['pr_auc'], 4)))
    print(rows[-1])
save_csv('b3_full_vs_edge.csv', rows)
print(f"size reduction x{rows[0]['size_KB'] / rows[1]['size_KB']:.0f} | p95 latency reduction x{rows[0]['p95_ms'] / rows[1]['p95_ms']:.0f}")
for r in rows:
    verdict = 'MEETS' if r['projected_edge_p95_ms'] <= LAT_BUDGET_MS and r['size_KB'] <= SIZE_BUDGET_KB else 'FAILS'
    print(f"{r['model']:<10}: projected device p95 {r['projected_edge_p95_ms']} ms (x{EDGE_SLOWDOWN} slowdown assumption), size {r['size_KB']} KB -> {verdict} the 50 ms / {SIZE_BUDGET_KB} KB budget")
seen_rec = lambda s: ((s >= .5) & (yte == 1) & seen_mask).sum() / ((yte == 1) & seen_mask).sum()
unseen_rec = lambda s: ((s >= .5) & (yte == 1) & ~seen_mask).sum() / ((yte == 1) & ~seen_mask).sum()
print(f"recall seen/unseen attack types: full {seen_rec(s_full):.3f}/{unseen_rec(s_full):.3f} | edge {seen_rec(s_edge):.3f}/{unseen_rec(s_edge):.3f}")
print(f"false-positive rate on normal traffic: full {m_full['confusion'][0][1] / sum(m_full['confusion'][0]):.4f} | edge {m_edge['confusion'][0][1] / sum(m_edge['confusion'][0]):.4f}")

k= 8 trees=10 depth= 8: size   20.2 KB | validation F1 0.9857


k= 8 trees=20 depth=10: size   81.4 KB | validation F1 0.9919


k= 8 trees=25 depth=12: size  140.8 KB | validation F1 0.9942


k=12 trees=10 depth= 8: size   19.9 KB | validation F1 0.9900


k=12 trees=20 depth=10: size   79.2 KB | validation F1 0.9929


k=12 trees=25 depth=12: size  154.1 KB | validation F1 0.9950


k=20 trees=10 depth= 8: size   19.8 KB | validation F1 0.9901


k=20 trees=20 depth=10: size   71.8 KB | validation F1 0.9941


k=20 trees=25 depth=12: size  143.2 KB | validation F1 0.9954
chosen on VALIDATION within the 64 KB budget: {'k': 20, 'trees': 10, 'depth': 8, 'size_KB': 19.8046875, 'val_f1': 0.9901372212692968}


features kept: ['src_bytes', 'dst_bytes', 'same_srv_rate', 'diff_srv_rate', 'logged_in', 'flag=SF', 'dst_host_diff_srv_rate', 'dst_host_srv_serror_rate', 'count', 'dst_host_same_srv_rate', 'dst_host_srv_count', 'srv_serror_rate', 'protocol_type=icmp', 'dst_host_same_src_port_rate', 'serror_rate', 'flag=S0', 'service=http', 'srv_count', 'dst_host_srv_diff_host_rate', 'dst_host_count']
float16 thresholds / 8-bit leaves agree with the float32 model on 99.92% of test decisions


{'model': 'full RF', 'size_KB': 28964.8, 'mean_ms': 15.3609, 'p95_ms': 21.0901, 'projected_edge_p95_ms': 105.451, 'batch_throughput_per_s': 50593, 'accuracy': 0.77, 'precision': 0.9681, 'recall': 0.6163, 'f1': 0.7531, 'roc_auc': 0.961, 'pr_auc': 0.9645}
{'model': 'edge model', 'size_KB': 20.7, 'mean_ms': 0.0348, 'p95_ms': 0.0444, 'projected_edge_p95_ms': 0.222, 'batch_throughput_per_s': 773296, 'accuracy': 0.7839, 'precision': 0.9665, 'recall': 0.6427, 'f1': 0.772, 'roc_auc': 0.9589, 'pr_auc': 0.9657}
size reduction x1399 | p95 latency reduction x475
full RF   : projected device p95 105.451 ms (x5.0 slowdown assumption), size 28964.8 KB -> FAILS the 50 ms / 64 KB budget
edge model: projected device p95 0.222 ms (x5.0 slowdown assumption), size 20.7 KB -> MEETS the 50 ms / 64 KB budget
recall seen/unseen attack types: full 0.769/0.247 | edge 0.774/0.324
false-positive rate on normal traffic: full 0.0269 | edge 0.0295


## Task B3.3 - PySpark RDD pipeline (Task A4 design)

In [22]:
%%writefile u5_spark.py
# Task A4 / B3.3 - RDD pipeline: per-host alert statistics from raw connection records (NSL-KDD lines),
# batch-scored with the compressed edge model. Same shape as the A4 air-quality pipeline:
# (key, timestamp, value) -> per-key mean and maximum -> ten "noisiest" keys.
import time, numpy as np

N_HOSTS = 5000


CHUNK = 50000


def make_generator(lines_bc, ranges_bc):
    """Narrow source: partition p synthesises records [start, end) by resampling raw NSL-KDD test lines and
    tagging each with a host id and a timestamp. Randomness is seeded per fixed-size chunk of the global
    record index, so the data set is identical whatever the number of partitions."""
    def gen(pidx, _it):
        lines = lines_bc.value
        start, end = ranges_bc.value[pidx]
        for c in range(start // CHUNK, (end - 1) // CHUNK + 1 if end > start else 0):
            rng = np.random.default_rng(1000 + c)
            idx = rng.integers(0, len(lines), CHUNK); hosts = rng.integers(0, N_HOSTS, CHUNK)
            lo, hi = max(start, c * CHUNK), min(end, (c + 1) * CHUNK)
            for g in range(lo, hi):
                k = g - c * CHUNK
                yield f'{hosts[k]},{g},{lines[idx[k]]}'
    return gen


def parse(line):
    """'host,ts,<41 KDD fields>,label,difficulty' -> (host, ts, fields)."""
    p = line.split(',')
    return int(p[0]), int(p[1]), p[2:43]


def make_featuriser(vocab_bc, numeric_idx, log_idx):
    def featurise(rec):
        host, ts, f = rec
        vocab = vocab_bc.value
        x = [float(f[i]) for i in numeric_idx]
        for j in log_idx: x[j] = float(np.log1p(x[j]))
        for c, col in ((1, 'protocol_type'), (2, 'service'), (3, 'flag')):
            oh = [0.0] * len(vocab[col])
            k = vocab[col].get(f[c])
            if k is not None: oh[k] = 1.0
            x += oh
        return host, ts, x
    return featurise


def make_scorer(model_bc):
    def score_partition(it):
        model = model_bc.value
        batch = []
        for rec in it:
            batch.append(rec)
            if len(batch) == 20000:
                yield from _score(model, batch); batch = []
        if batch: yield from _score(model, batch)
    return score_partition


def _score(model, batch):
    X = np.array([r[2] for r in batch], dtype=np.float32)
    s = model.score(X)
    for (host, ts, _), v in zip(batch, s):
        yield host, (1, int(v >= 0.5), float(v), float(v))          # (count, alerts, sum score, max score)


def merge(a, b):
    return a[0] + b[0], a[1] + b[1], a[2] + b[2], max(a[3], b[3])


def build_pipeline(sc, lines, enc, model, n_records, partitions, cache=False):
    lines_bc = sc.broadcast(lines)
    bounds = [n_records * i // partitions for i in range(partitions + 1)]
    per_bc = sc.broadcast([(bounds[i], bounds[i + 1]) for i in range(partitions)])
    vocab_bc = sc.broadcast({c: {v: i for i, v in enumerate(enc.vocab[c])} for c in enc.vocab})
    numeric_idx = [i for i in range(41) if i not in (1, 2, 3)]
    log_idx = [enc.numeric.index(c) for c in enc.LOG]
    model_bc = sc.broadcast(model)
    raw = sc.parallelize(range(partitions), partitions).mapPartitionsWithIndex(make_generator(lines_bc, per_bc))
    records = raw.map(parse)                                         # narrow
    feats = records.map(make_featuriser(vocab_bc, numeric_idx, log_idx))   # narrow
    if cache == 'feats': feats.cache()                               # persist BEFORE children are defined,
    scored = feats.mapPartitions(make_scorer(model_bc))              # narrow  (PySpark otherwise fuses the
    if cache == 'scored': scored.cache()                             #          narrow chain into one task)
    per_host = scored.reduceByKey(merge, numPartitions=partitions)   # WIDE: shuffle by host
    stats = per_host.mapValues(lambda v: (v[0], v[1], v[1] / v[0], v[2] / v[0], v[3]))  # narrow
    return feats, scored, per_host, stats


def cached_mb(sc):
    infos = sc._jsc.sc().getRDDStorageInfo()
    return sum(i.memSize() for i in infos) / 2 ** 20


def run_once(sc, lines, enc, model, n, P, cache=False):
    """cache: False, 'scored' (small (host, 4-number) records re-used by both actions) or 'feats'
    (122-float feature vectors: large, and only the scoring step reads them)."""
    feats, scored, per_host, stats = build_pipeline(sc, lines, enc, model, n, P, cache)
    t0 = time.perf_counter()
    total_alerts = scored.map(lambda kv: kv[1][1]).sum()             # action 1
    t1 = time.perf_counter()
    top10 = stats.takeOrdered(10, key=lambda kv: -kv[1][1])          # action 2 (re-uses scored)
    t2 = time.perf_counter()
    shuffles = stats.toDebugString().decode().count('ShuffledRDD')
    mem = cached_mb(sc)
    for r in list(sc._jsc.getPersistentRDDs().values()): r.unpersist()
    return dict(n=n, partitions=P, cache=cache or 'none', action1_s=t1 - t0, action2_s=t2 - t1, total_s=t2 - t0,
                alerts=int(total_alerts), shuffles=shuffles, cached_MB=mem, top10=top10)

Writing u5_spark.py


In [23]:
from u5_spark import *

In [24]:
os.environ['PYTHONPATH'] = os.getcwd()
from pyspark import SparkConf, SparkContext
sc = SparkContext(conf=SparkConf().setMaster('local[2]').setAppName('csa17-u5').set('spark.driver.memory', '3g')
                  .set('spark.ui.showConsoleProgress', 'false').set('spark.ui.enabled', 'false'))
sc.setLogLevel('ERROR')
lines = [l.strip() for l in open('data/KDDTest+.txt')]
print('Spark', sc.version, '| master local[2] (2 cores)')
feats_, scored_, ph_, stats_ = build_pipeline(sc, lines, enc, edge, 10 ** 5, 2)
print('lineage of the final RDD (narrow steps are fused into one PythonRDD; one ShuffledRDD = one shuffle):')
print(stats_.toDebugString().decode())
results = []
for n in (10 ** 5, 10 ** 6, 10 ** 7):
    for P in (1, 2, 4, 8):
        r = run_once(sc, lines, enc, edge, n, P); results.append(r)
        print(f"n={n:>9,} partitions={P}  action1 {r['action1_s']:7.2f} s  action2 {r['action2_s']:7.2f} s  total {r['total_s']:7.2f} s  "
              f"alerts {r['alerts']:>9,}  shuffles {r['shuffles']}", flush=True)
for n, P, c in [(10 ** 6, 2, 'scored'), (10 ** 6, 2, 'feats'), (10 ** 7, 2, 'scored')]:
    r = run_once(sc, lines, enc, edge, n, P, cache=c); results.append(r)
    print(f"n={n:>9,} partitions={P} cache={c:<7} action1 {r['action1_s']:7.2f} s  action2 {r['action2_s']:7.2f} s  total {r['total_s']:7.2f} s  cached {r['cached_MB']:.1f} MB", flush=True)
print('ten noisiest hosts at n=10^7 (host, (records, alerts, alert rate, mean score, max score)):')
for h in [r for r in results if r['n'] == 10 ** 7][0]['top10']: print('  ', h[0], tuple(round(v, 3) for v in h[1]))
save_csv('b3_spark_scaling.csv', [{k: v for k, v in r.items() if k != 'top10'} for r in results])

Spark 4.3.0 | master local[2] (2 cores)


lineage of the final RDD (narrow steps are fused into one PythonRDD; one ShuffledRDD = one shuffle):
(2) PythonRDD[5] at RDD at PythonRDD.scala:59 []
 |  MapPartitionsRDD[4] at mapPartitions at PythonRDD.scala:197 []
 |  ShuffledRDD[3] at partitionBy at DirectMethodHandleAccessor.java:103 []
 +-(2) PairwiseRDD[2] at reduceByKey at /home/claude/u5/nb/u5_spark.py:87 []
    |  PythonRDD[1] at reduceByKey at /home/claude/u5/nb/u5_spark.py:87 []
    |  ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:326 []


n=  100,000 partitions=1  action1    3.16 s  action2    2.59 s  total    5.76 s  alerts    37,858  shuffles 1


n=  100,000 partitions=2  action1    2.18 s  action2    1.30 s  total    3.48 s  alerts    37,858  shuffles 1


n=  100,000 partitions=4  action1    1.19 s  action2    1.85 s  total    3.03 s  alerts    37,858  shuffles 1


n=  100,000 partitions=8  action1    1.88 s  action2    1.95 s  total    3.82 s  alerts    37,858  shuffles 1


... (8 outputs left out)

n=1,000,000 partitions=2 cache=scored  action1    9.44 s  action2    0.50 s  total    9.94 s  cached 10.0 MB


n=1,000,000 partitions=2 cache=feats   action1   14.46 s  action2    6.94 s  total   21.40 s  cached 131.1 MB


n=10,000,000 partitions=2 cache=scored  action1  100.87 s  action2    4.76 s  total  105.63 s  cached 100.0 MB


ten noisiest hosts at n=10^7 (host, (records, alerts, alert rate, mean score, max score)):
   1862 (2060, 862, 0.418, 0.415, 1.0)
   434 (2116, 857, 0.405, 0.406, 1.0)
   4970 (2063, 851, 0.413, 0.41, 1.0)
   202 (2098, 849, 0.405, 0.401, 1.0)
   1637 (2128, 848, 0.398, 0.406, 1.0)
   1681 (2123, 848, 0.399, 0.397, 1.0)
   301 (2137, 848, 0.397, 0.398, 1.0)
   2497 (2050, 843, 0.411, 0.404, 1.0)
   367 (2031, 843, 0.415, 0.417, 1.0)
   2166 (2041, 843, 0.413, 0.412, 1.0)


## Task B4 - operator assistant and deployment decision

In [25]:
%%writefile u5_chatbot.py
# Task B4 - retrieval-based operator assistant for the intrusion detector (TF-IDF + cosine similarity,
# confidence-threshold fallback for out-of-scope requests) and a simulated speech-recognition noise channel.
import time, random, numpy as np

INTENTS = {
    'open_alerts': ['which alerts are open', 'show me the open alerts', 'list unresolved alerts', 'any active alerts right now',
                    'what alerts are still pending', 'give me all open incidents', 'are there alerts nobody has handled',
                    'display current alerts', 'open alerts please', 'what is currently flagged', 'outstanding alerts', 'alerts waiting for review', 'tickets that are not closed', 'whats open on the board', 'unacknowledged alarms', 'show me what still needs attention'],
    'explain_alert': ['why was this flagged', 'explain alert 3', 'why did the model raise this alert', 'what made connection 12 suspicious',
                      'reason for this detection', 'why is host 2041 marked as an attack', 'explain why this was classified malicious',
                      'what features triggered the alarm', 'justify this alert', 'why do you think this is an intrusion', 'on what basis was it flagged', 'which signals made it look malicious', 'break down the decision for this alert', 'what evidence supports this alarm', 'how did the model decide this', 'tell me what looked wrong in that traffic'],
    'alert_count': ['how many alerts today', 'number of alerts in the last hour', 'count the alerts', 'how many detections so far',
                    'total alerts this shift', 'how many attacks were detected', 'alert count please', 'how many incidents since morning',
                    'tell me the number of alarms', 'how busy has the detector been', 'alarm tally', 'how many have fired', 'volume of alerts since midnight', 'how many times did it alarm', 'give me the alert numbers', 'alerts raised in total'],
    'top_hosts': ['which hosts are the noisiest', 'top ten suspicious hosts', 'which machines generate most alerts', 'worst offending hosts',
                  'show hosts with the highest alert rate', 'who is attacking us the most', 'list the most flagged sources',
                  'rank hosts by alerts', 'which ip has the most alarms', 'most suspicious devices on the network', 'offenders leaderboard', 'which endpoints are causing trouble', 'biggest sources of alarms', 'most active suspicious addresses', 'machines to investigate first', 'heaviest hitters on the network'],
    'attack_types': ['what kinds of attacks did we see', 'which attack families were detected', 'breakdown of attack types',
                     'are these denial of service or probes', 'what categories of intrusion today', 'show attack distribution',
                     'types of threats observed', 'is it mostly dos traffic', 'which attack classes are most common', 'summarise the attacks by type', 'what sort of malicious traffic', 'group the alerts by category', 'scans or floods', 'which threat classes appear', 'mix of attack kinds', 'what is the attack profile'],
    'model_status': ['is the detector running', 'model health check', 'is the intrusion model online', 'status of the detection engine',
                     'is everything working', 'when was the model last updated', 'is the edge device alive', 'check system status',
                     'is the classifier up', 'detector heartbeat', 'are you connected to the sensor', 'is detection live', 'uptime of the detector', 'is the pipeline healthy', 'any errors on the device', 'is monitoring active'],
    'performance': ['how accurate is the model', 'what is the detection latency', 'how fast does it score a connection',
                    'what is the false alarm rate', 'model precision and recall', 'how good is the detector', 'what is the f1 score',
                    'how many milliseconds per sample', 'does it meet the latency budget', 'how reliable are the alerts', 'how often is it wrong', 'what is the recall', 'speed of inference', 'can we trust its alarms', 'error rate of the classifier', 'response time of the model'],
    'mark_false_positive': ['mark alert 5 as false positive', 'this alert is a false alarm', 'close alert 7 as benign',
                            'that was legitimate traffic', 'dismiss this alert', 'flag as false positive', 'alert 9 is not an attack',
                            'resolve this alert as harmless', 'it was a false alarm close it', 'report a false detection', 'that one is a mistake', 'whitelist this alert', 'benign activity close the ticket', 'label it as not malicious', 'wrong alarm please clear', 'false detection on that one'],
    'block_host': ['block host 2041', 'quarantine that machine', 'add this ip to the blocklist', 'cut off host 77',
                   'isolate the attacker', 'deny traffic from this source', 'block the suspicious host', 'firewall this address',
                   'disconnect host 1503', 'stop traffic from that device', 'ban that address', 'drop all packets from it', 'shut that machine out', 'cut the connection from that source', 'put it behind the firewall', 'blacklist the source'],
    'help': ['what can you do', 'help', 'list your commands', 'how do i use you', 'what questions can i ask',
             'show me the options', 'what are your features', 'i need help', 'menu', 'guide me', 'instructions please', 'what is supported', 'how should i talk to you', 'capabilities', 'usage', 'i am lost'],
}
TEST = {   # paraphrases written separately from the training phrases
    'open_alerts': ['what alerts are open now', 'show unresolved alarms', 'list the pending incidents', 'anything still open', 'current open detections', 'which alarms are active'],
    'explain_alert': ['why was alert 4 raised', 'explain this detection to me', 'what caused this flag', 'why is this connection suspicious', 'give the reason for alert 11', 'why did you flag host 300'],
    'alert_count': ['how many alarms so far today', 'count of detections this hour', 'total number of alerts', 'how many intrusions detected', 'alerts count for today', 'number of incidents this shift'],
    'top_hosts': ['top suspicious machines', 'which hosts raise most alarms', 'noisiest sources on the network', 'rank the most flagged hosts', 'which ip addresses are worst', 'hosts with most detections'],
    'attack_types': ['what attack types are we seeing', 'distribution of attack families', 'are they probes or dos attacks', 'kinds of intrusions detected', 'attack categories breakdown', 'what classes of attack occurred'],
    'model_status': ['is the model up and running', 'health of the detector', 'is the engine online', 'system status check', 'is the device working', 'detector status'],
    'performance': ['what is the model accuracy', 'scoring latency per connection', 'false alarm rate of the detector', 'precision of the model', 'is it fast enough', 'how good are its predictions'],
    'mark_false_positive': ['mark alert 2 as a false alarm', 'that alert is benign close it', 'false positive on alert 6', 'dismiss alert 8 as harmless', 'this was normal traffic', 'close this as false positive'],
    'block_host': ['block that ip address', 'quarantine host 812', 'put host 45 on the blocklist', 'isolate this device', 'deny the attacker', 'firewall host 2041'],
    'help': ['what are you able to do', 'show your commands', 'how does this work', 'help me please', 'which questions are supported', 'options menu'],
}
OOS_DEV = ['what is the weather tomorrow', 'tell me a joke', 'who won the cricket match', 'order a pizza', 'play some music',
           'what time is it in london', 'translate hello to french', 'book a meeting room', 'what is the capital of japan',
           'how do i bake bread', 'recommend a movie', 'set an alarm for 6 am', 'what is my salary', 'open youtube', 'convert dollars to rupees']
OOS_TEST = ['will it rain today', 'sing a song', 'who is the prime minister', 'buy me a coffee', 'what is the stock price of apple',
            'how tall is mount everest', 'remind me to call mom', 'best restaurants near me', 'how do i reset my email password',
            'what is two plus two', 'tell me about black holes', 'schedule leave for friday', 'turn on the lights', 'cricket score please',
            'write a poem about rain', 'how many calories in rice', 'who wrote hamlet', 'what day is it', 'call a taxi', 'news headlines']


class RetrievalBot:
    """TF-IDF over character 3-5-grams within word boundaries (robust to recognition errors and to word-level
    overlap such as "what is the" that out-of-scope requests share with intents); an utterance is mapped to the
    intent of its most similar training phrase; below `threshold` it is out of scope."""
    def __init__(self, intents, threshold=0.3):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.phrases = [p for i in intents for p in intents[i]]
        self.labels = [i for i in intents for _ in intents[i]]
        self.char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True).fit(self.phrases)
        self.M = self._vec(self.phrases)
        self.threshold = threshold

    def _vec(self, texts):
        from sklearn.preprocessing import normalize
        return normalize(self.char.transform(texts))

    def scores(self, texts):
        S = (self._vec(texts) @ self.M.T).toarray()
        j = S.argmax(1)
        return [self.labels[k] for k in j], S.max(1)

    def classify(self, text, threshold=None):
        lab, conf = self.scores([text])
        th = self.threshold if threshold is None else threshold
        return (lab[0] if conf[0] >= th else 'out_of_scope'), float(conf[0])


def corrupt(text, wer, rng, vocab):
    """Speech-recognition noise: each word is substituted / deleted / followed by an inserted word with
    total probability `wer` (substitution 60 %, deletion 25 %, insertion 15 %)."""
    out = []
    for w in text.split():
        u = rng.random()
        if u < wer * 0.60: out.append(_sound_alike(w, rng, vocab))
        elif u < wer * 0.85: continue
        elif u < wer: out += [w, rng.choice(vocab)]
        else: out.append(w)
    return ' '.join(out) if out else rng.choice(vocab)


def _sound_alike(w, rng, vocab):
    """Recognisers usually confuse similar-sounding words: pick a vocabulary word sharing the first letter
    or a one-character edit of the word; fall back to a random word."""
    cands = [v for v in vocab if v != w and v[0] == w[0] and abs(len(v) - len(w)) <= 2]
    if cands and rng.random() < 0.7: return rng.choice(cands)
    if len(w) > 3:
        i = rng.randrange(len(w)); return w[:i] + rng.choice('aeioustnrl') + w[i + 1:]
    return rng.choice(vocab)


def evaluate(bot, wer=0.0, threshold=None, seed=0, reps=5):
    """In-scope intent accuracy, out-of-scope detection rate and false-accept rate (OOS accepted as an intent)."""
    vocab = sorted({w for t in list(bot.phrases) + OOS_DEV for w in t.split()})
    rng = random.Random(seed)
    th = bot.threshold if threshold is None else threshold
    ins = [(u, i) for i in TEST for u in TEST[i]]
    correct = total = oos_ok = oos_n = 0
    pairs = []
    for _ in range(reps):
        utts = [corrupt(u, wer, rng, vocab) for u, _ in ins]
        lab, conf = bot.scores(utts)
        for (u, i), l, c in zip(ins, lab, conf):
            p = l if c >= th else 'out_of_scope'
            correct += p == i; total += 1; pairs.append((i, p))
        o = [corrupt(u, wer, rng, vocab) for u in OOS_TEST]
        lab, conf = bot.scores(o)
        for c in conf:
            oos_ok += c < th; oos_n += 1
    return dict(wer=wer, threshold=th, intent_acc=correct / total, oos_detect=oos_ok / oos_n,
                false_accept=1 - oos_ok / oos_n, pairs=pairs)


def choose_threshold(intents, grid=np.round(np.arange(0.10, 0.71, 0.02), 2), held=3):
    """Pick the threshold without touching the test sets: refit on all but the last `held` phrases of each
    intent, score those held phrases (in scope) and OOS_DEV (out of scope), maximise in-scope accuracy +
    out-of-scope rejection. The final bot is then refitted on all training phrases."""
    fit = {k: v[:-held] for k, v in intents.items()}
    dev = [(u, k) for k, v in intents.items() for u in v[-held:]]
    bot = RetrievalBot(fit)
    lab, conf = bot.scores([u for u, _ in dev])
    ok = np.array([l == k for l, (_, k) in zip(lab, dev)])
    _, conf_oos = bot.scores(OOS_DEV)
    best, curve = None, []
    for th in grid:
        acc = float(np.mean(ok & (conf >= th))); rej = float(np.mean(conf_oos < th))
        curve.append((float(th), acc, rej))
        if best is None or acc + rej > best[1]: best = (float(th), acc + rej, acc, rej)
    return best, curve

Writing u5_chatbot.py


In [26]:
from u5_chatbot import *

In [27]:
from sklearn.metrics import classification_report, confusion_matrix
(th, _, dev_acc, dev_rej), th_curve = choose_threshold(INTENTS)
bot = RetrievalBot(INTENTS, th)
print(f"{len(INTENTS)} intents x {len(next(iter(INTENTS.values())))} training phrases | threshold {th} chosen on development phrases "
      f"(dev in-scope accuracy {dev_acc:.2f}, dev out-of-scope rejection {dev_rej:.2f})")
ev = evaluate(bot, 0.0, reps=1)
true = [t for t, _ in ev['pairs']]; predicted = [p for _, p in ev['pairs']]
names = list(INTENTS) + ['out_of_scope']
print(classification_report(true, predicted, labels=names[:-1], zero_division=0, digits=3))
cm = confusion_matrix(true + ['out_of_scope'] * len(OOS_TEST), predicted + [bot.classify(u)[0] for u in OOS_TEST], labels=names)
print('confusion matrix (rows = true intent, last row/column = out of scope):')
print(pd.DataFrame(cm, index=[n[:10] for n in names], columns=[n[:6] for n in names]).to_string())
ts = []
for u in [u for v in TEST.values() for u in v] * 20:
    t0 = time.perf_counter(); bot.classify(u); ts.append((time.perf_counter() - t0) * 1000)
print(f"response latency: mean {np.mean(ts):.3f} ms, p95 {np.percentile(ts, 95):.3f} ms over {len(ts)} queries")
alerts = te.assign(score=s_edge).head(400).query('score >= 0.5').head(5)
def answer(text):
    intent, conf = bot.classify(text)
    if intent == 'open_alerts': return f'{len(alerts)} open alerts: ' + ', '.join(f"#{i} {r.service}/{r.flag}" for i, r in alerts.iterrows())
    if intent == 'explain_alert':
        i = alerts.index[0]; x = Xte[i]; f = edge.features
        names_ = [enc.names[j] for j in f]
        top = sorted(zip(names_, x[f], sub.feature_importances_), key=lambda z: -z[2])[:3]
        return f'alert #{i} scored {s_edge[i]:.2f}; highest-importance inputs: ' + ', '.join(f'{n}={v:.2f}' for n, v, _ in top)
    if intent == 'performance': return f"edge model accuracy {m_edge['accuracy']:.3f}, F1 {m_edge['f1']:.3f}, p95 latency {lat_edge['p95_ms']:.3f} ms"
    if intent == 'out_of_scope': return f'Sorry, that is outside what I can do (confidence {conf:.2f}). Say "help" for options.'
    return f'[{intent}] handler'
for q in ['which alerts are open', 'why was this flagged', 'how fast is the detector', 'who wrote hamlet', 'will it rain today']:
    print(f'> {q}\n  {answer(q)}')

10 intents x 16 training phrases | threshold 0.4 chosen on development phrases (dev in-scope accuracy 0.43, dev out-of-scope rejection 0.87)
                     precision    recall  f1-score   support

        open_alerts      0.800     0.667     0.727         6
      explain_alert      1.000     0.500     0.667         6
        alert_count      0.714     0.833     0.769         6
          top_hosts      0.833     0.833     0.833         6
       attack_types      1.000     0.833     0.909         6
       model_status      1.000     1.000     1.000         6
        performance      1.000     1.000     1.000         6
mark_false_positive      1.000     0.833     0.909         6
         block_host      0.857     1.000     0.923         6
               help      1.000     0.833     0.909         6

          micro avg      0.909     0.833     0.870        60
          macro avg      0.920     0.833     0.865        60
       weighted avg      0.920     0.833     0.865        60

co

response latency: mean 0.968 ms, p95 1.176 ms over 1200 queries
> which alerts are open
  5 open alerts: #0 private/REJ, #1 private/REJ, #3 eco_i/SF, #12 private/REJ, #13 telnet/S0
> why was this flagged
  alert #0 scored 1.00; highest-importance inputs: src_bytes=0.00, same_srv_rate=0.04, dst_bytes=0.00
> how fast is the detector
  edge model accuracy 0.784, F1 0.772, p95 latency 0.044 ms
> who wrote hamlet
  Sorry, that is outside what I can do (confidence 0.31). Say "help" for options.
> will it rain today
  [alert_count] handler


In [28]:
rows = []
for w in (0.0, 0.1, 0.2, 0.3):
    r = evaluate(bot, w, reps=5, seed=1); r.pop('pairs'); rows.append({k: round(float(v), 4) for k, v in r.items()})
    print(f"WER {w:.0%}: intent accuracy {r['intent_acc']:.3f} | out-of-scope detected {r['oos_detect']:.3f} | false-accept rate {r['false_accept']:.3f}")
save_csv('b4_wer.csv', rows)
rng = random.Random(3); vocab = sorted({w for t in bot.phrases + OOS_DEV for w in t.split()})
for u in ['why was alert 4 raised', 'block that ip address']:
    print(f'   example at 30% WER: "{u}" -> "{corrupt(u, 0.3, rng, vocab)}"')
sweep = []
for w in (0.0, 0.2, 0.3):
    for t in np.round(np.arange(0.20, 0.66, 0.05), 2):
        r = evaluate(bot, w, threshold=t, reps=3, seed=2); sweep.append(dict(wer=w, threshold=t, intent_acc=r['intent_acc'], false_accept=r['false_accept']))
save_csv('b4_threshold_sweep.csv', sweep)
sw = pd.DataFrame(sweep)
print(sw.pivot(index='threshold', columns='wer', values=['intent_acc', 'false_accept']).round(3).to_string())
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for w, st in [(0.0, 'C0-o'), (0.2, 'C1-s'), (0.3, 'C3-^')]:
    d = sw[sw.wer == w]
    ax[0].plot(d.threshold, d.intent_acc, st, label=f'WER {int(w*100)}%'); ax[1].plot(d.false_accept, d.intent_acc, st, label=f'WER {int(w*100)}%')
ax[0].axvline(th, color='grey', ls=':', label=f'chosen {th}')
ax[0].set_xlabel('confidence threshold'); ax[0].set_ylabel('in-scope intent accuracy'); ax[0].legend(); ax[0].set_title('Threshold sweep')
ax[1].set_xlabel('false-accept rate (out-of-scope accepted)'); ax[1].set_ylabel('in-scope intent accuracy'); ax[1].legend(); ax[1].set_title('Accuracy against false accepts')
plt.tight_layout(); plt.savefig('figures/b4_threshold.png', dpi=130); plt.show()

WER 0%: intent accuracy 0.833 | out-of-scope detected 0.850 | false-accept rate 0.150
WER 10%: intent accuracy 0.777 | out-of-scope detected 0.760 | false-accept rate 0.240
WER 20%: intent accuracy 0.737 | out-of-scope detected 0.690 | false-accept rate 0.310
WER 30%: intent accuracy 0.630 | out-of-scope detected 0.700 | false-accept rate 0.300
   example at 30% WER: "why was alert 4 raised" -> "was alert 4 raised"
   example at 30% WER: "block that ip address" -> "board that from address"


          intent_acc               false_accept              
wer              0.0    0.2    0.3          0.0    0.2    0.3
threshold                                                    
0.20           0.883  0.811  0.672         0.70  0.833  0.800
0.25           0.883  0.811  0.672         0.65  0.733  0.767
0.30           0.883  0.806  0.672         0.55  0.667  0.650
0.35           0.850  0.778  0.650         0.25  0.433  0.467
0.40           0.833  0.728  0.606         0.15  0.317  0.267
0.45           0.750  0.644  0.533         0.10  0.233  0.233
0.50           0.667  0.533  0.439         0.05  0.133  0.167
0.55           0.533  0.417  0.339         0.00  0.083  0.150
0.60           0.450  0.317  0.217         0.00  0.017  0.100
0.65           0.317  0.228  0.189         0.00  0.017  0.083


In [29]:
RTT_MS = (20, 80, 300)          # assumed round trip to a cloud region: good 4G, congested 4G, rural 2G/3G
ALERTS_PER_DAY = int(m_edge['confusion'][0][1] + m_edge['confusion'][1][1]) * 10 ** 7 // len(yte)
print('Evidence for the edge-versus-cloud decision (detector):')
print(f"  edge model: p95 {lat_edge['p95_ms']:.3f} ms here, x{EDGE_SLOWDOWN} -> {EDGE_SLOWDOWN * lat_edge['p95_ms']:.2f} ms on device; size {edge.nbytes() / 1024:.1f} KB")
print(f"  full model in the cloud: p95 {lat_full['p95_ms']:.2f} ms compute + network RTT {RTT_MS} ms -> " +
      ', '.join(f"{lat_full['p95_ms'] + r:.0f} ms" for r in RTT_MS))
print(f"  accuracy: full {m_full['accuracy']:.4f} / ROC-AUC {m_full['roc_auc']:.4f} vs edge {m_edge['accuracy']:.4f} / ROC-AUC {m_edge['roc_auc']:.4f}")
print(f"  edge batch throughput {int(thr_edge):,} records/s (one core) vs full {int(thr_full):,} records/s")
raw_bytes = 10 ** 7 * 120
print(f"  privacy/bandwidth: shipping 10^7 connection records/day to the cloud ~ {raw_bytes / 1e9:.1f} GB/day of metadata; "
      f"shipping only alerts (~{ALERTS_PER_DAY:,}/day at the test-set alert rate) ~ {ALERTS_PER_DAY * 200 / 1e6:.0f} MB/day")
print('Evidence for the assistant: response p95 ' + f"{np.percentile(ts, 95):.3f} ms; model size " +
      f"{sum(len(p) for p in bot.phrases) / 1024:.1f} KB of phrases + TF-IDF vocabulary {len(bot.char.vocabulary_):,} n-grams")

Evidence for the edge-versus-cloud decision (detector):
  edge model: p95 0.044 ms here, x5.0 -> 0.22 ms on device; size 20.7 KB
  full model in the cloud: p95 21.09 ms compute + network RTT (20, 80, 300) ms -> 41 ms, 101 ms, 321 ms
  accuracy: full 0.7700 / ROC-AUC 0.9610 vs edge 0.7839 / ROC-AUC 0.9589
  edge batch throughput 773,296 records/s (one core) vs full 50,593 records/s
  privacy/bandwidth: shipping 10^7 connection records/day to the cloud ~ 1.2 GB/day of metadata; shipping only alerts (~3,785,486/day at the test-set alert rate) ~ 757 MB/day
Evidence for the assistant: response p95 1.176 ms; model size 3.9 KB of phrases + TF-IDF vocabulary 2,888 n-grams
